# Avance Fase 3 — Semana 2

## Núcleo algorítmico y programación orientada a objetos

**Caso de trabajo: Comodidades, capacidad y ubicación vs. precio y ocupación de los alojamientos Airbnb en Santiago de Chile**

---

En la Fase 2 ustedes construyeron un pipeline que funciona. Este cuaderno lo toma y le
da forma de **sistema**: las mismas operaciones, ahora organizadas en clases que se
pueden probar, reemplazar y reutilizar.

No hay datos nuevos ni pipeline nuevo. **Lo que cambia es la arquitectura del código.**

### Recorrido

| Parte | Contenido | Criterio de la rúbrica que alimenta |
|---|---|---|
| 1 | De funciones a objetos: por qué | Codificación funcional |
| 2 | Los cinco conceptos de la POO | Programación orientada a objetos |
| 3 | La clase `Preprocesador` | Preprocesamiento y transformación |
| 4 | Encapsulamiento: proteger el estado | Programación orientada a objetos |
| 5 | Herencia y polimorfismo | Programación orientada a objetos |
| 6 | El `Pipeline`: componer los pasos | Diseño estructurado |
| 7 | Cohesión y acoplamiento | Diseño estructurado |
| 8 | Validación: casos normales, límite y excepciones | Validación técnica |
| 9 | Recursividad | Diseño estructurado |
| 10 | Eficiencia: medir tiempo y memoria | Eficiencia y optimización |
| 11 | Patrones de diseño | Documentación de arquitectura |
| 12 | Adaptar el cuaderno a su propio conjunto | Codificación funcional |
| 13 | El resultado: código y datos listos para la Fase 4 | Documentación de arquitectura |
| 14 | Ejercicios para su proyecto | Todos |

**Cómo usarlo.** Los datos se ingresan **desde fuera**: el cuaderno lee el archivo que
ustedes indiquen en la celda de configuración. Para trabajar con el conjunto de su
proyecto basta con editar esa celda; el resto funciona igual.

**Qué cambió respecto de la Fase 2.** Los pasos de S1_F2_Preprocesamiento.ipynb quedan encapsulados en clases que heredan de Transformador. La sección 8 comprueba que el resultado es idéntico a data/processed/airbnb_procesado.csv.

---
## Configuración

**Esta es la única celda que hay que editar.** Todo lo que sigue se construye a partir
de ella, así que para trabajar con el conjunto de su propio proyecto basta con cambiar
estos valores.

In [1]:
# =====================================================================
# CONFIGURACIÓN DEL PROYECTO
# Editen solo esta celda para trabajar con su propio conjunto de datos
# =====================================================================

# Ruta escrita desde la raíz del proyecto, no desde la carpeta de este cuaderno
RUTA_DATOS = "data/raw/datasetairbnb.csv"         # archivo crudo
RUTA_F2 = "data/processed/airbnb_procesado.csv"   # resultado de la F2, para comparar

COLUMNA_ID = "id"
OBJETIVOS = ["price", "estimated_occupancy_l365d"]   # no se imputan ni se escalan

VARIABLES_ESTUDIO = ["price", "estimated_occupancy_l365d", "amenities", "accommodates",
                     "bedrooms", "beds", "neighbourhood_cleansed", "latitude",
                     "host_is_superhost", "minimum_nights", "maximum_nights"]

COLUMNAS_IMPUTAR = ["accommodates", "latitude", "bedrooms", "beds",
                    "minimum_nights", "maximum_nights"]
COLUMNAS_MARCAR = ["bedrooms"]
COLUMNAS_NOMINALES = {"host_is_superhost": "superhost",
                      "neighbourhood_cleansed": "comuna"}
COLUMNAS_ESCALAR = ["accommodates", "latitude", "bedrooms", "beds",
                    "minimum_nights", "maximum_nights", "n_comodidades"]

COMODIDADES = {
    "tiene_aire_acondicionado": r"air conditioning|\bac\b",
    "tiene_estacionamiento":    r"parking",
    "tiene_piscina":            r"\bpool\b(?! table)",
    "tiene_lavadora":           r"\bwasher\b",
    "tiene_gimnasio":           r"\bgym\b",
    "tiene_ascensor":           r"elevator",
    "tiene_espacio_trabajo":    r"dedicated workspace",
    "tiene_self_checkin":       r"self check-in",
}

UMBRAL_PRECIO = 10_000_000
MAX_NOCHES = 1125
SEMILLA = 42
PROPORCION_PRUEBA = 0.2

COLUMNAS_ESPERADAS = [COLUMNA_ID, "room_type"] + VARIABLES_ESTUDIO

# ==== Rutas: esta parte es igual que antes ====
from pathlib import Path

def localizar_raiz(inicio=None):
    """Si el cuaderno está en F3, la raíz del proyecto es la carpeta de arriba."""
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio

RAIZ = localizar_raiz()
RUTA_DATOS = str(RAIZ / RUTA_DATOS)
RUTA_F2 = str(RAIZ / RUTA_F2)
DIR_PROCESADO = str(RAIZ / "data" / "processed")

print("Archivo crudo :", RUTA_DATOS)
print("Resultado F2  :", RUTA_F2)
print("Objetivos     :", OBJETIVOS)

Archivo crudo : c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\raw\datasetairbnb.csv
Resultado F2  : c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\airbnb_procesado.csv
Objetivos     : ['price', 'estimated_occupancy_l365d']


## Preparación del entorno

In [2]:
import os
import sys
import time
import timeit
import tracemalloc

import numpy as np
import pandas as pd

SEMILLA = 42
np.random.seed(SEMILLA)

print("Python :", sys.executable)
print("pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Semilla:", SEMILLA)

Python : c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\.venv\Scripts\python.exe
pandas : 3.1.1
NumPy  : 2.5.4
Semilla: 42


## Carga del conjunto desde el archivo externo

El cuaderno **no genera los datos**: los lee del archivo que ustedes indiquen en la
configuración. La función de carga hace tres cosas antes de devolver nada:

1. Comprueba que el archivo exista, y si no, explica dónde ponerlo.
2. Lo lee según su extensión, sea CSV o Excel.
3. Verifica que estén las columnas declaradas en la configuración.

Ese tercer paso es el que evita el error más molesto: descubrir en la celda veinte que
una columna se llamaba distinto.

In [3]:
def leer_archivo(ruta):
    """Lee el archivo según su extensión y devuelve un DataFrame."""
    extension = os.path.splitext(ruta)[1].lower()

    if extension in (".csv", ".txt"):
        # sep=None con engine="python" infiere el separador: útil con datos
        # públicos chilenos, que suelen venir con punto y coma
        return pd.read_csv(ruta, sep=None, engine="python", encoding="utf-8")
    if extension in (".xlsx", ".xls"):
        return pd.read_excel(ruta)
    if extension == ".parquet":
        return pd.read_parquet(ruta)

    raise ValueError(
        f"Extensión no reconocida: '{extension}'. "
        "Se admiten .csv, .txt, .xlsx, .xls y .parquet."
    )


def verificar_esquema(df, columnas_esperadas):
    """Comprueba que estén todas las columnas declaradas en la configuración."""
    faltantes = [c for c in columnas_esperadas if c not in df.columns]
    if faltantes:
        raise KeyError(
            f"Faltan columnas declaradas en la configuración: {faltantes}\n"
            f"Columnas disponibles en el archivo: {list(df.columns)}"
        )
    sobrantes = [c for c in df.columns if c not in columnas_esperadas]
    return {"declaradas": len(columnas_esperadas),
            "en_archivo": df.shape[1],
            "no_declaradas": sobrantes}

In [4]:
def cargar(ruta=RUTA_DATOS, columnas_esperadas=None):
    """Carga el archivo crudo y verifica que estén las columnas del pipeline."""
    columnas_esperadas = columnas_esperadas or COLUMNAS_ESPERADAS

    # Si el archivo no está, se detiene con un mensaje que dice qué hacer
    if not os.path.exists(ruta):
        raise FileNotFoundError(
            f"No se encontró el archivo: {ruta}\n"
            "Descárguenlo según el README y guárdenlo como data/raw/datasetairbnb.csv."
        )
    df = leer_archivo(ruta)
    print(f"Archivo leído: {ruta}")

    # Revisa que estén las 13 columnas que usa el pipeline
    informe = verificar_esquema(df, columnas_esperadas)
    print(f"Forma: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(f"Esquema verificado: las {informe['declaradas']} columnas del pipeline están presentes")
    return df


datos = cargar()
datos.head()

Archivo leído: c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\raw\datasetairbnb.csv
Forma: 18904 filas x 90 columnas
Esquema verificado: las 13 columnas del pipeline están presentes


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,49392,https://www.airbnb.com/rooms/49392,20260923225049,2026-09-26,NaN,Share my Flat in Providencia,<b>The space</b><br />Departamento en el barri...,NaN,https://a0.muscache.com/pictures/3740612/b1850...,224592,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN
1,52811,https://www.airbnb.com/rooms/52811,20260923225049,2026-09-26,NaN,Suite Providencia 1 Santiago Chile,Apartment located on the subway station Manuel...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,244792,...,4.59,4.64,4.36,NaN,NaN,3,3,0,0,0.23
2,80482,https://www.airbnb.com/rooms/80482,20260923225049,2026-09-25,NaN,Room Private for Woman,"Near transport, subway, malls supermarkets, wa...",NaN,https://a0.muscache.com/pictures/3633153/dffd5...,154527,...,NaN,NaN,NaN,NaN,NaN,2,0,2,0,NaN
3,88944,https://www.airbnb.com/rooms/88944,20260923225049,2026-09-26,NaN,COZY APT. PROVIDENCIA METRO WIFI TV,Spacious and cozy apartment. It has excellent ...,NaN,https://a0.muscache.com/pictures/8fd1c60d-5a8f...,485358,...,4.89,4.96,4.73,NaN,NaN,1,1,0,0,1.37
4,90694,https://www.airbnb.com/rooms/90694,20260923225049,2026-09-26,NaN,Apartment x Rent in Providencia 802,APARTMENT<br />Furnished Apartments with one b...,NaN,https://a0.muscache.com/pictures/d29b8d52-ca46...,491253,...,4.83,4.77,4.43,NaN,NaN,2,2,0,0,0.41


**Si obtienen un `KeyError` al cargar**, la configuración no coincide con el archivo.
El mensaje lista las columnas que faltan y las que sí están: corrijan los nombres en la
celda de configuración y vuelvan a ejecutar.


## El conjunto antes de tocarlo

Un perfil que funciona con cualquier conjunto, no solo con este. Se apoya en la
configuración, así que al cambiar de proyecto sigue sirviendo.

In [5]:
def perfilar(df):
    """Devuelve una fila por columna con su tipo, valores únicos y porcentaje de nulos."""
    return pd.DataFrame({
        "columna": df.columns,
        "tipo": [str(t) for t in df.dtypes],
        "unicos": [df[c].nunique(dropna=True) for c in df.columns],
        "nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().mean() * 100).round(2).values,
    }).sort_values("pct_nulos", ascending=False).reset_index(drop=True)


perfil = perfilar(datos)

print("Columnas con valores faltantes:")
print(perfil[perfil["nulos"] > 0].to_string(index=False))

print("\nTop 10 columnas con más valores nulos:")
print(perfil[["columna", "pct_nulos"]].head(10).to_string(index=False))

print("\nDistribución del tipo de alojamiento:")
print(datos["room_type"].value_counts())

print("\nDistribución del tipo de propiedad:")
print(datos["property_type"].value_counts().head(10))

print("\nObjetivos del estudio:", OBJETIVOS)
print("Ojo: en el crudo 'price' viene como texto, por ejemplo:", datos["price"].dropna().iloc[0])
print(datos["estimated_occupancy_l365d"].describe())

Columnas con valores faltantes:
                     columna    tipo  unicos  nulos  pct_nulos
       neighborhood_overview float64       0  18904     100.00
                  host_since float64       0  18904     100.00
          host_response_time float64       0  18904     100.00
          host_thumbnail_url float64       0  18904     100.00
        host_acceptance_rate float64       0  18904     100.00
          host_response_rate float64       0  18904     100.00
          host_verifications float64       0  18904     100.00
               neighbourhood float64       0  18904     100.00
   host_total_listings_count float64       0  18904     100.00
          host_neighbourhood float64       0  18904     100.00
            calendar_updated float64       0  18904     100.00
            instant_bookable float64       0  18904     100.00
neighbourhood_group_cleansed float64       0  18904     100.00
                   bathrooms float64       0  18904     100.00
                     li

**Tres observaciones que condicionan todo lo que viene (documentadas en la Fase 2).**

1. `price` viene como texto (`$55,106.00`): hay que convertirlo a número antes de cualquier cálculo.
2. Los objetivos no se imputan. Rellenar `price` o `estimated_occupancy_l365d` sería inventar la respuesta: los alojamientos sin precio se eliminan.
3. Hay valores imposibles: precios sobre $10.000.000, `maximum_nights` de 2.147.483.647 y anuncios con más dormitorios que huéspedes.

Ninguna decisión se rehace aquí: la Fase 3 las encapsula en clases.

---
# 1. De funciones a objetos: por qué

Así se ve el código de la Fase 2, en celdas sueltas:

In [6]:
# El estilo de la Fase 2: funciona, pero cada celda depende de las anteriores
df_f2 = datos.copy()
df_f2 = df_f2.drop(columns=["id"])
df_f2["beds"] = df_f2["beds"].fillna(df_f2["beds"].median())
df_f2 = pd.get_dummies(df_f2, columns=["room_type"], dtype=int)

print("Resultado (Filas x Columnas):", df_f2.shape)

Resultado (Filas x Columnas): (18904, 92)


Funciona. Pero tiene cuatro problemas que aparecen cuando el proyecto crece:

| Problema | Consecuencia |
|---|---|
| El orden importa y no está declarado | Ejecutar una celda fuera de orden produce otro resultado |
| No se puede reutilizar sin copiar | El mismo código se duplica en el cuaderno de F3 y F4 |
| No se puede probar por partes | Si el resultado está mal, hay que revisar todo |
| La mediana se calcula sobre todo el conjunto | Si después separan entrenamiento y prueba, hay **fuga de datos** |

La programación orientada a objetos resuelve los cuatro: cada paso pasa a ser una pieza
con nombre, con estado propio y con una interfaz fija.

---
# 2. Programación orientada a objetos desde cero

Esta parte es para quienes nunca han trabajado con clases. Si ya conocen el paradigma,
pueden pasar a la parte 3, pero conviene mirar el trazado de la celda 2.3: explica algo
que casi nadie enseña y que aclara `self` de una vez.

## 2.1 La idea: un molde y sus copias

Una **clase** es un molde. No es un dato: es la descripción de qué datos guarda algo y
qué operaciones sabe hacer.

Un **objeto** es una copia concreta hecha con ese molde. De un mismo molde se pueden
sacar muchas copias, y cada una guarda sus propios valores.

Una analogía que funciona: la clase es el formulario en blanco y el objeto es un
formulario llenado. El formulario en blanco define qué campos hay; cada formulario
llenado tiene sus propios valores en esos campos.

En ciencia de datos lo usamos cuando un componente necesita **recordar algo** entre una
llamada y otra. Ese es el criterio para decidir entre función y clase.

In [7]:
# Una clase mínima, con todo lo que hay que entender
class AlojamientosAirbnb:
    """Gestiona y acumula las reservas de un alojamiento en Santiago.

    El docstring, este texto entre comillas triples, es la documentación de la
    clase. Aparece al escribir help(AlojamientosAirbnb).
    """

    def __init__(self, precio_por_noche):
        # __init__ es el CONSTRUCTOR: Python lo ejecuta solo al crear el objeto.
        # Su trabajo es dejar listos los atributos iniciales.
        # self es el objeto (alojamiento específico) que se está creando.
        self.precio_noche = precio_por_noche  # ATRIBUTO: precio fijo por noche
        self.noches_reservadas = 0             # ATRIBUTO: contador de noches acumuladas

    def agregar_reserva(self, cantidad_noches):
        # MÉTODO: una operación que el objeto sabe hacer (registrar noches).
        # Recibe self porque necesita saber sobre CUÁL alojamiento opera.
        self.noches_reservadas = self.noches_reservadas + cantidad_noches
        return self.noches_reservadas

    def calcular_ingresos_totales(self):
        # MÉTODO: calcula los ingresos multiplicando las noches por el precio base
        return self.noches_reservadas * self.precio_noche


# Crear un objeto: acá Python ejecuta __init__ por nosotros
depto_providencia = AlojamientosAirbnb(precio_por_noche=60000)
print("Precio por noche configurado:", depto_providencia.precio_noche)

depto_providencia.agregar_reserva(3)        # Reserva de 3 noches
depto_providencia.agregar_reserva(5)        # Reserva de 5 noches más

print("Noches totales reservadas:", depto_providencia.noches_reservadas)
print("Ingresos totales acumulados (CLP):", depto_providencia.calcular_ingresos_totales())
print("Tipo del objeto:", type(depto_providencia))

Precio por noche configurado: 60000
Noches totales reservadas: 8
Ingresos totales acumulados (CLP): 480000
Tipo del objeto: <class '__main__.AlojamientosAirbnb'>


## 2.2 Cada objeto tiene sus propios datos

Este es el punto que más cuesta al principio: dos objetos de la misma clase **no
comparten** sus atributos.

In [8]:
depto_providencia = AlojamientosAirbnb(precio_por_noche=60000)  # primer alojamiento
depto_las_condes = AlojamientosAirbnb(precio_por_noche=95000)    # segundo alojamiento, independiente del primero

depto_providencia.agregar_reserva(3)  # 3 noches reservadas en Providencia
depto_las_condes.agregar_reserva(5)    # 5 noches reservadas en Las Condes

print("Noches en Providencia:", depto_providencia.noches_reservadas)
print("Noches en Las Condes:", depto_las_condes.noches_reservadas)
print("¿Son el mismo objeto?", depto_providencia is depto_las_condes)
print("\nCada alojamiento guarda lo suyo. El molde (la clase) es el mismo; los datos y precios, no.")

# Lo que cada objeto guarda internamente se puede ver directamente con __dict__
print("\nContenido interno de depto_providencia:", depto_providencia.__dict__)
print("Contenido interno de depto_las_condes:", depto_las_condes.__dict__)

Noches en Providencia: 3
Noches en Las Condes: 5
¿Son el mismo objeto? False

Cada alojamiento guarda lo suyo. El molde (la clase) es el mismo; los datos y precios, no.

Contenido interno de depto_providencia: {'precio_noche': 60000, 'noches_reservadas': 3}
Contenido interno de depto_las_condes: {'precio_noche': 95000, 'noches_reservadas': 5}


## 2.3 Qué es `self`, explicado con el trazado real

`self` es la respuesta a una pregunta práctica: si el método (como registrar nches o calcular ingresos) está escrito una sola vez en la clase, **¿cómo sabe sobre cuál objeto trabajar?**

La respuesta es que Python le pasa el objeto como primer argumento. Cuando ustedes
escriben `depto_providencia.agregar_reserva(3)`, Python ejecuta por detrás `AlojamientosAirbnb.agregar_reserva(depto_providencia, 3)`.

Eso significa que `self` no tiene nada de mágico: es simplemente el primer parámetro, y
podría llamarse de otra forma. Se llama `self` por convención, y conviene respetarla.

In [9]:
depto_nunoa = AlojamientosAirbnb(precio_por_noche=50000)

# Las dos líneas siguientes hacen exactamente lo mismo
depto_nunoa.agregar_reserva(7)                 # forma normal (el objeto viaja solo)
AlojamientosAirbnb.agregar_reserva(depto_nunoa, 7)       # lo que Python ejecuta por detrás de forma explícita

print("Noches totales tras las dos llamadas equivalentes:", depto_nunoa.noches_reservadas)
print("\nPor eso todo método lleva self como primer parámetro:")
print("es el espacio donde llega el objeto sobre el que se opera.")

Noches totales tras las dos llamadas equivalentes: 14

Por eso todo método lleva self como primer parámetro:
es el espacio donde llega el objeto sobre el que se opera.


## 2.4 Dónde busca Python un atributo

Cuando escriben `objeto.algo`, Python lo busca en este orden:

1. En el **objeto**, es decir en su `__dict__`.
2. Si no está, en su **clase**.
3. Si no está, en la **clase madre**, y así hacia arriba.

Esa cadena se llama MRO, por *method resolution order*, y se puede consultar. Entenderla
explica por qué la herencia funciona.

In [10]:
class AlojamientoAirbnb:
    tipo_plataforma = "Airbnb"          # Atributo de CLASE: uno solo, compartido por todos

    def __init__(self, comuna, precio_noche):
        self.comuna = comuna            # Atributo de INSTANCIA: único para cada depto
        self.precio_noche = precio_noche # Atributo de INSTANCIA: único para cada depto


depto_1 = AlojamientoAirbnb("Providencia", 60000)
depto_2 = AlojamientoAirbnb("Las Condes", 90000)

print("Atributos propios de depto_1 :", depto_1.__dict__)
print("Atributo de la clase          :", AlojamientoAirbnb.tipo_plataforma)
print("depto_1.tipo_plataforma ->", depto_1.tipo_plataforma, "(no está en la instancia: Python sube a la clase)")

# Cambiar el atributo de clase afecta a TODOS los objetos
AlojamientoAirbnb.tipo_plataforma = "Airbnb Global"
print("\nDespués de cambiarlo en la clase:")
print("  depto_1.tipo_plataforma :", depto_1.tipo_plataforma)
print("  depto_2.tipo_plataforma:", depto_2.tipo_plataforma)

# Cambiar un atributo de instancia afecta solo a ese objeto específico
depto_1.precio_noche = 65000
print("\nDespués de actualizar solo el precio de depto_1:")
print("  depto_1.precio_noche :", depto_1.precio_noche)
print("  depto_2.precio_noche:", depto_2.precio_noche)

print("\nCadena de búsqueda (MRO):", [c.__name__ for c in AlojamientoAirbnb.__mro__])

Atributos propios de depto_1 : {'comuna': 'Providencia', 'precio_noche': 60000}
Atributo de la clase          : Airbnb
depto_1.tipo_plataforma -> Airbnb (no está en la instancia: Python sube a la clase)

Después de cambiarlo en la clase:
  depto_1.tipo_plataforma : Airbnb Global
  depto_2.tipo_plataforma: Airbnb Global

Después de actualizar solo el precio de depto_1:
  depto_1.precio_noche : 65000
  depto_2.precio_noche: 90000

Cadena de búsqueda (MRO): ['AlojamientoAirbnb', 'object']


**La conclusión práctica.** Un atributo de clase sirve para constantes compartidas, como
una etiqueta o un valor por defecto. Un atributo de instancia sirve para lo que cambia
de un objeto a otro, que es casi todo lo que nos interesa en un pipeline.

## 2.5 Resumen de los cinco conceptos

| Concepto | Qué es | Cómo se reconoce en el código |
|---|---|---|
| **Clase** | El molde | `class NombreDeLaClase:` |
| **Objeto** | Una copia del molde | `obj = NombreDeLaClase(...)` |
| **Atributo** | Un dato que el objeto guarda | `self.algo = valor` |
| **Método** | Una operación del objeto | `def hacer(self, ...):` |
| **`self`** | El objeto sobre el que se opera | Primer parámetro de todo método |

Con esos cinco alcanza para leer todo lo que sigue.

---
# 3. La clase `Preprocesador`

Primera versión: una clase que guarda la tabla y sabe prepararla. Cada método hace
**una sola cosa** y devuelve `self`, lo que permite encadenar llamadas.

In [11]:
class Preprocesador:
    """Guarda la tabla de Airbnb y sabe prepararla para el análisis.

    La tabla vive en self.df. Cada método es un paso del pipeline que trabaja
    sobre esa misma tabla y devuelve self, para poder encadenar.
    """

    def __init__(self, df):
        self.df = df.copy()          # copia: no se modifica el original
        self.registro = []           # lo que el objeto va recordando

    def quitar_identificador(self):
        """El id no aporta información al análisis: identifica, no describe."""
        if "id" in self.df.columns:
            self.df = self.df.drop(columns=["id"])
            self.registro.append("id eliminado")
        return self

    def imputar_puntajes(self):
        """Rellena los nulos de beds con la mediana."""
        if "beds" in self.df.columns:
            nulos = int(self.df["beds"].isna().sum())
            mediana = float(self.df["beds"].median())
            self.df["beds"] = self.df["beds"].fillna(mediana)
            self.registro.append(f"beds: {nulos} nulos imputados con mediana={mediana:.2f}")
        return self

    def codificar(self, columnas):
        """Convierte columnas categóricas en columnas binarias (0/1)."""
        columnas_validas = [c for c in columnas if c in self.df.columns]
        if columnas_validas:
            antes = self.df.shape[1]
            self.df = pd.get_dummies(self.df, columns=columnas_validas, dtype=int)
            self.registro.append(f"codificadas {columnas_validas}: {antes} -> {self.df.shape[1]} columnas")
        return self

    def informe(self):
        """Devuelve lo que el objeto recuerda haber hecho."""
        return "\n".join(f"  {i}. {paso}" for i, paso in enumerate(self.registro, 1))


# Ejemplo de uso encadenado (Method Chaining)
prep = Preprocesador(datos)
prep.quitar_identificador().imputar_puntajes().codificar(["room_type", "neighbourhood_cleansed"])

print("Resultado final (Filas x Columnas):", prep.df.shape)
print("\nLo que el objeto recuerda:")
print(prep.informe())

Resultado final (Filas x Columnas): (18904, 123)

Lo que el objeto recuerda:
  1. id eliminado
  2. beds: 203 nulos imputados con mediana=2.00
  3. codificadas ['room_type', 'neighbourhood_cleansed']: 89 -> 123 columnas


**Lo que ganamos con esto.** El objeto `prep` recuerda qué hizo y con qué valores. En la
Fase 2 esa información había que anotarla a mano en la bitácora; acá es producto de la
ejecución y no puede quedar desactualizada.

**Lo que todavía falta.** Esta clase mezcla dos cosas: aprender de los datos y
aplicarlos. Eso se arregla en la parte siguiente.

---
# 4. Encapsulamiento: proteger el estado interno

El `Preprocesador` de arriba tiene un problema serio: calcula la mediana **sobre el
conjunto completo**. Si después separan entrenamiento y prueba, la mediana ya vio los
datos de prueba. Eso se llama **fuga de datos** y es uno de los errores más caros en
ciencia de datos.

La solución es separar dos momentos:

- **`ajustar`**: aprende los parámetros, y solo del conjunto de entrenamiento.
- **`transformar`**: los aplica a cualquier conjunto.

Y proteger ese estado interno con un guion bajo al inicio del nombre, que en Python
significa «esto es interno, no lo toques desde fuera».

In [12]:
class ImputadorCamas:
    """Imputa beds separando lo que aprende de lo que aplica."""

    def __init__(self):
        self._mediana = None        # interno: se aprende, no se asigna desde fuera
        self._ajustado = False      # interno: controla el orden de las llamadas

    @property
    def ajustado(self):
        """Solo lectura: se puede consultar, no asignar."""
        return self._ajustado

    @property
    def mediana(self):
        return self._mediana

    def ajustar(self, df):
        self._mediana = float(df["beds"].median())
        self._ajustado = True
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError(
                "ImputadorCamas: hay que llamar a ajustar() antes de transformar(). "
                "La mediana se aprende del conjunto de entrenamiento."
            )
        df = df.copy()
        df["beds"] = df["beds"].fillna(self._mediana)
        return df


# El orden equivocado ahora falla con un mensaje claro
imputador = ImputadorCamas()
try:
    imputador.transformar(datos)
except RuntimeError as error:
    print("RuntimeError:", error)

RuntimeError: ImputadorCamas: hay que llamar a ajustar() antes de transformar(). La mediana se aprende del conjunto de entrenamiento.


In [13]:
# El orden correcto, con separación de conjuntos
entrenamiento = datos.sample(frac=1 - PROPORCION_PRUEBA, random_state=SEMILLA)
prueba = datos.drop(index=entrenamiento.index)

imputador = ImputadorCamas().ajustar(entrenamiento)
prueba_lista = imputador.transformar(prueba)

print("Mediana aprendida del entrenamiento:", round(imputador.mediana, 2))
print("Mediana del conjunto de prueba     :", round(prueba["beds"].median(), 2))
print("Nulos en prueba después de imputar :", int(prueba_lista["beds"].isna().sum()))
print("\nLa mediana aplicada a prueba es la del entrenamiento, aunque en este caso")
print("ambas coincidan: prueba nunca se usó para aprender nada.")

# La propiedad es de solo lectura
try:
    imputador.ajustado = False
except AttributeError as error:
    print("\nAttributeError:", error)

Mediana aprendida del entrenamiento: 2.0
Mediana del conjunto de prueba     : 2.0
Nulos en prueba después de imputar : 0

La mediana aplicada a prueba es la del entrenamiento, aunque en este caso
ambas coincidan: prueba nunca se usó para aprender nada.

AttributeError: property 'ajustado' of 'ImputadorCamas' object has no setter


**Por qué importa.** Sin el control de `_ajustado`, transformar antes de ajustar
produciría un resultado sin sentido **y sin ningún mensaje de error**. El
encapsulamiento convierte un error silencioso en uno visible.

En el informe, este bloque es la evidencia del encapsulamiento: atributos internos con
guion bajo, propiedades de solo lectura y validación antes de actuar.

---
# 5. Herencia y polimorfismo

Miren el `ImputadorCamas`: el control de `_ajustado`, la copia defensiva y el mensaje de
error van a repetirse en el codificador, en el escalador (por ejemplo, para estandarizar el precio de los alojamientos) y en cualquier otro paso de procesamiento del dataset Airbnb.

**Herencia** es escribir eso una sola vez en una clase base.

In [14]:
class Transformador:
    """Clase base: lo que todos los pasos del pipeline de Airbnb comparten.

    Esta clase NO se usa directamente. Su trabajo es definir el contrato:
    qué métodos tendrá todo paso del pipeline y cómo se controla su estado.
    Las clases hijas solo completan lo que cambia de un paso a otro.
    """

    def __init__(self, columna):
        # Atributo público: cualquiera puede leerlo y cambiarlo
        self.columna = columna

        # Atributos con guion bajo: por convención, INTERNOS.
        # Python no lo impide, pero el guion bajo le dice a quien lee el código
        # que no debe tocarlos desde fuera de la clase.
        self._parametros = {}       # lo que el paso aprende del conjunto
        self._ajustado = False      # controla que no se transforme antes de ajustar

    # @property convierte un método en algo que se lee como atributo:
    # se escribe paso.nombre y no paso.nombre(). Y como no hay setter,
    # el valor NO se puede asignar desde fuera. Eso es encapsulamiento.
    def _columnas(self):
        """Devuelve siempre una lista, aunque se haya indicado una sola columna."""
        return [self.columna] if isinstance(self.columna, str) else list(self.columna)

    @property
    def nombre(self):
        # type(self).__name__ devuelve el nombre de la clase REAL del objeto
        etiqueta = self.columna if isinstance(self.columna, str) else ", ".join(self.columna)
        return f"{type(self).__name__}({etiqueta})"

    @property
    def parametros(self):
        # dict(...) crea una COPIA. Si devolviéramos self._parametros
        # directamente, quien lo recibe podría modificar el estado interno
        # del objeto sin que la clase se entere. A esto se le llama
        # copia defensiva.
        return dict(self._parametros)

    def ajustar(self, df):
        """Aprende los parámetros del conjunto que recibe."""
        # 1) Validar la entrada ANTES de trabajar: si algo falta, se avisa
        #    acá y no veinte líneas más abajo con un error incomprensible
        faltantes = [c for c in self._columnas() if c not in df.columns]
        if faltantes:
            raise KeyError(f"{self.nombre}: no existen las columnas {faltantes}.")

        # 2) Delegar el cálculo a la clase hija. La base no sabe QUÉ se aprende;
        #    solo sabe CUÁNDO hay que aprenderlo.
        self._parametros = self.aprender(df)

        # 3) Registrar que ya se ajustó, para poder controlarlo después
        self._ajustado = True

        # 4) Devolver self permite encadenar: paso.ajustar(df).transformar(df)
        return self

    def transformar(self, df):
        """Aplica la transformación usando lo aprendido en ajustar()."""
        # Esta guarda es el corazón del encapsulamiento: impide un uso
        # incorrecto que de otro modo pasaría inadvertido
        if not self._ajustado:
            raise RuntimeError(f"{self.nombre}: hay que ajustar antes de transformar.")

        # df.copy() evita modificar el DataFrame que nos pasaron. Sin esto,
        # el objeto original del usuario cambiaría sin que él lo pidiera.
        return self.aplicar(df.copy())

    def ajustar_transformar(self, df):
        """Atajo para el caso habitual: aprender y aplicar sobre el mismo conjunto."""
        return self.ajustar(df).transformar(df)

    # ---- Métodos que cada clase hija DEBE implementar -------------------
    # Se definen acá lanzando NotImplementedError para dejar el contrato
    # explícito: si una hija olvida implementarlos, el error lo dice claro.

    def aprender(self, df):
        """Calcula y devuelve los parámetros. Lo implementa cada clase hija."""
        raise NotImplementedError("Cada clase hija debe implementar aprender().")

    def aplicar(self, df):
        """Aplica la transformación. Lo implementa cada clase hija."""
        raise NotImplementedError("Cada clase hija debe implementar aplicar().")


# La clase base no se usa sola
try:
    Transformador("beds").ajustar(datos)
except NotImplementedError as error:
    print("NotImplementedError:", error)

NotImplementedError: Cada clase hija debe implementar aprender().


In [15]:
class ImputadorMediana(Transformador):
    """Rellena los nulos con la mediana aprendida.

    El paréntesis (Transformador) es la HERENCIA: esta clase recibe todo lo
    que tiene Transformador sin volver a escribirlo. Solo implementa los dos
    métodos que le faltaban.
    """

    def aprender(self, df):
        return {"mediana": float(df[self.columna].median()),
                "nulos_en_ajuste": int(df[self.columna].isna().sum())}

    def aplicar(self, df):
        df[self.columna] = df[self.columna].fillna(self._parametros["mediana"])
        return df


class CodificadorNominal(Transformador):
    """Convierte una columna de categorías en columnas 0/1.

    El vocabulario se aprende en el ajuste: una categoría que solo aparece en
    prueba no genera columna nueva, porque el modelo no pudo aprender de ella.
    """
    def __init__(self, columna, prefijo=None):
        super().__init__(columna)          # primero, lo que hace la clase base
        # El prefijo da nombre a las columnas nuevas: "comuna" -> comuna_Providencia
        self.prefijo = prefijo or columna
    def aprender(self, df):
        # Se guarda la lista de categorías ORDENADA, para que el resultado sea
        # el mismo en cada ejecución. Sin sorted(), el orden podría variar.
        return {"categorias": sorted(df[self.columna].dropna().unique())}

    def aplicar(self, df):
        # Se recorre el vocabulario aprendido, NO las categorías de este df.
        # Por eso una categoría nueva en prueba no genera columna: el modelo
        # nunca la vio y no podría haber aprendido nada de ella.
        for categoria in self._parametros["categorias"]:
            # Los nombres de columna no deben llevar espacios ni guiones
            etiqueta = str(categoria).strip().replace(" ", "_").replace("-", "_")
            # La comparación devuelve True/False; astype(int) lo pasa a 1/0
            df[f"{self.prefijo}_{etiqueta}"] = (df[self.columna] == categoria).astype(int)

        # La columna original ya no aporta: su información quedó en las nuevas
        return df.drop(columns=[self.columna])


class EscaladorRobusto(Transformador):
    """Centra en la mediana y divide por el rango intercuartílico (RobustScaler).

    Es el escalador que eligió la Fase 2: la mediana y el RIC casi no cambian
    por unos pocos valores extremos, así que esos extremos no deciden la escala.
    """

    def aprender(self, df):
        q1, mediana, q3 = df[self.columna].quantile([0.25, 0.5, 0.75])
        ric = float(q3 - q1)
        # Si el RIC es 0 (columna casi constante) se divide por 1: evita dividir por cero
        return {"mediana": float(mediana), "ric": ric if ric != 0 else 1.0}

    def aplicar(self, df):
        df[self.columna] = ((df[self.columna] - self._parametros["mediana"])
                            / self._parametros["ric"])
        return df

class EliminadorColumnas(Transformador):
    """Quita columnas que no aportan al análisis, como el identificador."""

    def aprender(self, df):
        return {"existe": self.columna in df.columns}

    def aplicar(self, df):
        return df.drop(columns=[self.columna]) if self.columna in df.columns else df


imp = ImputadorMediana("beds")
salida = imp.ajustar_transformar(datos)
print(imp.nombre, "->", imp.parametros)
print("Nulos en beds después:", int(salida["beds"].isna().sum()))

ImputadorMediana(beds) -> {'mediana': 2.0, 'nulos_en_ajuste': 203}
Nulos en beds después: 0


**Se realiza la transformación de la variable** "price", eliminando interferencias que hacen ver al valor como str, transformando de esta manera a float.

In [16]:
import json


class ConversorPrecio(Transformador):
    """Convierte el precio de texto ("$55,106.00") a número."""

    def aprender(self, df):
        return {"regla": "quitar '$' y ',' y convertir a número"}

    def aplicar(self, df):
        texto = (df[self.columna].astype(str)
                 .str.replace("$", "", regex=False)
                 .str.replace(",", "", regex=False)
                 .str.strip())
        df[self.columna] = pd.to_numeric(texto, errors="coerce")
        return df


class FiltroFilas(Transformador):
    """Conserva solo las filas que cumplen una condición."""

    def __init__(self, columna, condicion, descripcion):
        super().__init__(columna)
        self.condicion = condicion         # una función que dice True/False por fila
        self.descripcion = descripcion

    @property
    def nombre(self):
        # Sobrescritura: el nombre muestra la regla, que dice más que la columna
        return f"FiltroFilas({self.descripcion})"

    def aprender(self, df):
        return {"regla": self.descripcion}

    def aplicar(self, df):
        return df[self.condicion(df)].copy()


class SelectorColumnas(Transformador):
    """Conserva solo las variables del estudio, en el orden declarado."""

    @property
    def nombre(self):
        return f"SelectorColumnas({len(self._columnas())} variables)"

    def aprender(self, df):
        return {"conservadas": len(self._columnas())}

    def aplicar(self, df):
        return df[self._columnas()].copy()


class Acotador(Transformador):
    """Acota una columna a un máximo."""

    def __init__(self, columna, maximo):
        super().__init__(columna)
        self.maximo = maximo

    def aprender(self, df):
        return {"maximo": self.maximo,
                "valores_sobre_el_maximo": int((df[self.columna] > self.maximo).sum())}

    def aplicar(self, df):
        df[self.columna] = df[self.columna].clip(upper=self.maximo)
        return df


class MarcadorNulos(Transformador):
    """Bandera 0/1 de qué filas tenían el dato faltante. Va ANTES de imputar."""

    def aprender(self, df):
        return {"nulos_en_ajuste": int(df[self.columna].isna().sum())}

    def aplicar(self, df):
        df[f"{self.columna}_imputado"] = df[self.columna].isna().astype(int)
        return df


class ExtractorComodidades(Transformador):
    """Pasa la lista de texto 'amenities' a un conteo y a indicadores 0/1."""

    def __init__(self, columna, patrones):
        super().__init__(columna)
        self.patrones = dict(patrones)

    def aprender(self, df):
        return {"indicadores": list(self.patrones)}

    def aplicar(self, df):
        listas = df[self.columna].apply(json.loads)               # texto -> lista real
        texto = listas.apply(lambda lista: "|".join(lista).lower())
        df["n_comodidades"] = listas.str.len()
        for nueva, patron in self.patrones.items():
            df[nueva] = texto.str.contains(patron, regex=True).astype(int)
        return df.drop(columns=[self.columna])


# Prueba rápida: el precio deja de ser texto
conversor = ConversorPrecio("price")
convertido = conversor.ajustar_transformar(datos)
print(conversor.nombre, "->", datos["price"].dropna().iloc[0], "pasa a",
      convertido["price"].dropna().iloc[0])

ConversorPrecio(price) -> $59,303.00 pasa a 59303.0


### Cómo comprobar que la herencia funciona

No hay que creer en la herencia: se puede verificar.

In [17]:
imputador = ImputadorMediana("beds")

# 1) El objeto ES un ImputadorMediana Y TAMBIÉN es un Transformador
print("¿Es ImputadorMediana?", isinstance(imputador, ImputadorMediana))
print("¿Es Transformador?   ", isinstance(imputador, Transformador))

# 2) La cadena de búsqueda muestra de dónde hereda
print("\\nCadena de herencia:", [c.__name__ for c in ImputadorMediana.__mro__])

# 3) Métodos que NO están escritos en la clase hija y sin embargo funcionan,
#    porque se heredaron de la clase base
print("\\nMétodos heredados de Transformador:")
for metodo in ["ajustar", "transformar", "ajustar_transformar"]:
    definido_en = "ImputadorMediana" if metodo in ImputadorMediana.__dict__ else "Transformador"
    print(f"  {metodo:<22} definido en {definido_en}")

print("\\nMétodos propios de la clase hija:")
for metodo in ["aprender", "aplicar"]:
    print(f"  {metodo:<22} definido en ImputadorMediana")

¿Es ImputadorMediana? True
¿Es Transformador?    True
\nCadena de herencia: ['ImputadorMediana', 'Transformador', 'object']
\nMétodos heredados de Transformador:
  ajustar                definido en Transformador
  transformar            definido en Transformador
  ajustar_transformar    definido en Transformador
\nMétodos propios de la clase hija:
  aprender               definido en ImputadorMediana
  aplicar                definido en ImputadorMediana


**Lo que acaban de ver.** `ImputadorMediana` tiene ocho líneas de código propio y
dispone de toda la maquinaria de control de estado. Eso es lo que la herencia ahorra: si
mañana hay que cambiar el mensaje de error, se cambia en un solo lugar y las cuatro
clases hijas lo heredan.

**Polimorfismo** es que quien usa estas clases no necesita saber de qué tipo es cada
una: todas ofrecen `ajustar` y `transformar`.

In [18]:
pasos = [
    EliminadorColumnas("id"),
    ImputadorMediana("beds"),
    CodificadorNominal("room_type"),
    CodificadorNominal("neighbourhood_cleansed"),
    EscaladorRobusto("minimum_nights")
]

resultado = datos.copy()
for paso in pasos:                      # la misma llamada polimórfica para todos los pasos
    resultado = paso.ajustar_transformar(resultado)
    print(f"{paso.nombre:<45} -> {resultado.shape}")

EliminadorColumnas(id)                        -> (18904, 89)


ImputadorMediana(beds)                        -> (18904, 89)
CodificadorNominal(room_type)                 -> (18904, 92)
CodificadorNominal(neighbourhood_cleansed)    -> (18904, 123)
EscaladorRobusto(minimum_nights)              -> (18904, 123)


**Este bloque es la evidencia de los tres principios**, y así conviene declararlo en el
informe:

- **Herencia**: las cuatro clases heredan de `Transformador` y no repiten el control de estado.
- **Polimorfismo**: el bucle llama `ajustar_transformar()` sin preguntar el tipo.
- **Encapsulamiento**: `_parametros` y `_ajustado` son internos, y hay validación antes de actuar.

Y fíjense en la consecuencia práctica: **agregar un transformador nuevo no obliga a
tocar el bucle**. Eso es lo que la rúbrica llama bajo acoplamiento.

---
# 6. El `Pipeline`: componer los pasos

Una clase que guarda el orden, los ejecuta y controla que se haya ajustado antes de
transformar.

In [19]:
class Pipeline:
    """Encadena transformadores y los ejecuta en orden."""

    def __init__(self, pasos=None):
        self._pasos = list(pasos) if pasos else []
        self._ajustado = False

    def agregar(self, transformador):
        if not isinstance(transformador, Transformador):
            raise TypeError(
                f"Se esperaba un Transformador y se recibió {type(transformador).__name__}."
            )
        self._pasos.append(transformador)
        return self

    def ajustar(self, df):
        """Aprende los parámetros de cada paso SOLO con estos datos."""
        intermedio = df.copy()
        for paso in self._pasos:
            intermedio = paso.ajustar_transformar(intermedio)
        self._ajustado = True
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError("Pipeline: hay que ajustar antes de transformar.")
        resultado = df.copy()
        for paso in self._pasos:
            resultado = paso.transformar(resultado)
        return resultado

    def pasos_ejecutados(self):
        """Devuelve los pasos, para poder registrar sus parámetros aprendidos."""
        return tuple(self._pasos)

    def resumen(self):
        return pd.DataFrame([
            {"orden": i, "paso": p.nombre, "clase": type(p).__name__}
            for i, p in enumerate(self._pasos, start=1)
        ])

    def __len__(self):
        return len(self._pasos)

    def __repr__(self):
        estado = "ajustado" if self._ajustado else "sin ajustar"
        return f"Pipeline({len(self._pasos)} pasos, {estado})"


pipeline = Pipeline([
    EliminadorColumnas("id"),
    ImputadorMediana("beds"),
    CodificadorNominal("room_type"),
    CodificadorNominal("neighbourhood_cleansed"),
    EscaladorRobusto("minimum_nights"),
])

print(pipeline)
pipeline.resumen()

Pipeline(5 pasos, sin ajustar)


,orden,paso,clase
0,1,EliminadorColumnas(id),EliminadorColumnas
1,2,ImputadorMediana(beds),ImputadorMediana
2,3,CodificadorNominal(room_type),CodificadorNominal
3,4,CodificadorNominal(neighbourhood_cleansed),CodificadorNominal
4,5,EscaladorRobusto(minimum_nights),EscaladorRobusto


In [20]:
# Ajustar con entrenamiento, transformar prueba: sin fuga de datos
pipeline.ajustar(entrenamiento)
prueba_lista = pipeline.transformar(prueba)

print("Entrenamiento:", entrenamiento.shape, " -> Prueba transformada:", prueba_lista.shape)

escalador = pipeline.pasos_ejecutados()[-1]
print(f"\n{escalador.nombre} aprendió del entrenamiento: {escalador.parametros}")
print("Mediana de minimum_nights en prueba, ANTES de escalar:", prueba["minimum_nights"].median())
print("Esos parámetros se aplicaron a prueba sin recalcular nada con sus datos.")

Entrenamiento: (15123, 90)  -> Prueba transformada: (3781, 123)

EscaladorRobusto(minimum_nights) aprendió del entrenamiento: {'mediana': 1.0, 'ric': 1.0}
Mediana de minimum_nights en prueba, ANTES de escalar: 2.0
Esos parámetros se aplicaron a prueba sin recalcular nada con sus datos.


**Por qué importa.** El escalador no recalcula la mediana ni el RIC con los datos de prueba: aplica los que aprendió del entrenamiento. Si salieran de prueba, el análisis usaría información que no debía ver.

In [21]:
def construir_pipeline():
    """Arma el pipeline de la Fase 2 a partir de la configuración, en el MISMO orden."""
    pasos = [
        ConversorPrecio("price"),
        FiltroFilas("room_type", lambda d: d["room_type"] == "Entire home/apt",
                    "solo alojamientos completos"),
        SelectorColumnas(VARIABLES_ESTUDIO),
        Acotador("maximum_nights", MAX_NOCHES),
        FiltroFilas(OBJETIVOS, lambda d: d[OBJETIVOS].notna().all(axis=1),
                    "objetivos sin faltantes"),
    ]
    pasos += [MarcadorNulos(c) for c in COLUMNAS_MARCAR]
    pasos += [ImputadorMediana(c) for c in COLUMNAS_IMPUTAR]
    pasos += [
        FiltroFilas("price", lambda d: ~(d["price"] > UMBRAL_PRECIO),
                    f"precio <= {UMBRAL_PRECIO:,}"),
        FiltroFilas(["bedrooms", "accommodates"],
                    lambda d: ~(d["bedrooms"] > d["accommodates"]),
                    "dormitorios <= huéspedes"),
        ExtractorComodidades("amenities", COMODIDADES),
    ]
    pasos += [CodificadorNominal(c, prefijo) for c, prefijo in COLUMNAS_NOMINALES.items()]
    pasos += [EscaladorRobusto(c) for c in COLUMNAS_ESCALAR]
    return Pipeline(pasos)


pipeline_proyecto = construir_pipeline()
print(pipeline_proyecto)
pipeline_proyecto.resumen()

Pipeline(24 pasos, sin ajustar)


,orden,paso,clase
0,1,ConversorPrecio(price),ConversorPrecio
1,2,FiltroFilas(solo alojamientos completos),FiltroFilas
2,3,SelectorColumnas(11 variables),SelectorColumnas
3,4,Acotador(maximum_nights),Acotador
4,5,FiltroFilas(objetivos sin faltantes),FiltroFilas
5,6,MarcadorNulos(bedrooms),MarcadorNulos
6,7,ImputadorMediana(accommodates),ImputadorMediana
7,8,ImputadorMediana(latitude),ImputadorMediana
8,9,ImputadorMediana(bedrooms),ImputadorMediana
9,10,ImputadorMediana(beds),ImputadorMediana


---
# 7. Cohesión y acoplamiento

Son las dos palabras que más aparecen en el criterio de diseño estructurado y que casi
ningún material explica. Acá van con código.

**Alta cohesión** significa que cada componente hace **una sola cosa**.

**Bajo acoplamiento** significa que los componentes dependen poco unos de otros: cambiar
uno no obliga a tocar los demás.

## 7.1 Cohesión baja: una clase que hace de todo

Este es el diseño que aparece cuando se escribe rápido. Funciona, pero cada método toca
cosas distintas y la clase no tiene un propósito único.

In [22]:
class ProcesadorAirbnbTodoEnUno:
    """Ejemplo de COHESIÓN BAJA en el proyecto: hace cuatro cosas sin relación entre sí."""

    def __init__(self, ruta_csv):
        self.ruta_csv = ruta_csv
        self.df = None

    def cargar(self):            # Responsabilidad 1: entrada y salida de archivos (I/O)
        self.df = pd.read_csv(self.ruta_csv)

    def limpiar(self):           # Responsabilidad 2: transformar datos (Pipeline)
        if self.df is not None:
            self.df = self.df.dropna(subset=["beds"])

    def graficar(self):          # Responsabilidad 3: visualización (Matplotlib / Seaborn)
        # Generar gráficos de precios o distribución de alojamientos en Santiago
        pass

    def enviar_correo(self):     # Responsabilidad 4: comunicación / notificaciones
        # Enviar reporte de la auditoría de alojamientos por email
        pass


print("Cuatro responsabilidades distintas (cargar, limpiar, graficar, enviar correo) en una sola clase.")
print("Problema práctico: para probar la limpieza de los datos de Airbnb hay que tener un archivo real,")
print("porque cargar() y limpiar() viven pegadas en el mismo objeto.")

Cuatro responsabilidades distintas (cargar, limpiar, graficar, enviar correo) en una sola clase.
Problema práctico: para probar la limpieza de los datos de Airbnb hay que tener un archivo real,
porque cargar() y limpiar() viven pegadas en el mismo objeto.


## 7.2 Cohesión alta: cada clase con un propósito

El mismo trabajo, repartido. Ahora cada pieza se puede probar sola.

In [23]:
import pandas as pd
import numpy as np

class CargadorAirbnb:
    """Una sola responsabilidad: leer el archivo CSV con los alojamientos de Santiago."""

    def __init__(self, ruta):
        self.ruta = ruta

    def cargar(self):
        return pd.read_csv(self.ruta)


class LimpiadorAirbnb:
    """Una sola responsabilidad: limpiar nulos en métricas clave (como beds).

    Fíjense en que NO sabe de dónde vienen los datos. Recibe un DataFrame y
    devuelve otro. Eso es bajo acoplamiento: no depende del Cargador.
    """

    def limpiar(self, df):
        # Limpiar filas donde falte el puntaje de reseñas de los alojamientos
        df = df.copy()
        if "beds" in df.columns:
            df = df.dropna(subset=["beds"])
        return df


# La gran ventaja se ve al probar: no hace falta ningún archivo real ni CSV en disco
mini = pd.DataFrame({
    "id": [101, 102, 103],
    "beds": [4.5, np.nan, 4.8],
    "room_type": ["Entire home/apt", "Private room", "Entire home/apt"]
})

print("Se puede probar el LimpiadorAirbnb sin tocar el disco ni archivos pesados:")
limpiador = LimpiadorAirbnb()
print("  entrada:", mini.shape, " -> salida:", limpiador.limpiar(mini).shape)

Se puede probar el LimpiadorAirbnb sin tocar el disco ni archivos pesados:
  entrada: (3, 3)  -> salida: (2, 3)


## 7.3 La prueba del acoplamiento

Hay una pregunta que resuelve el asunto sin teoría:

> **Si cambiamos la forma de imputar, ¿cuántos archivos hay que modificar?**

Si la respuesta es **uno**, el acoplamiento es bajo. Si son tres, el diseño es frágil.

Veámoslo con el pipeline que ya construimos.

In [24]:
# Cambiar un paso del pipeline no obliga a tocar ni el Pipeline ni los otros pasos
pipeline_a = Pipeline([EliminadorColumnas("id"), ImputadorMediana("beds")])
pipeline_b = Pipeline([EliminadorColumnas("id"), ImputadorMediana("minimum_nights")])

for nombre, pipe in [("versión A", pipeline_a), ("versión B", pipeline_b)]:
    salida = pipe.ajustar(datos).transformar(datos)
    print(f"{nombre}: {salida.shape}")

print("\nSe cambió el paso y NO se modificó la clase Pipeline ni las demás clases.")
print("Eso es bajo acoplamiento, y es lo que el criterio de diseño evalúa.")

versión A: (18904, 89)
versión B: (18904, 89)

Se cambió el paso y NO se modificó la clase Pipeline ni las demás clases.
Eso es bajo acoplamiento, y es lo que el criterio de diseño evalúa.


## 7.4 Cómo se traduce en la estructura de archivos

Una organización con alta cohesión y bajo acoplamiento se reconoce a simple vista:

```
src/
├── carga.py            solo lee y verifica archivos
├── transformadores.py  solo las clases de transformación
├── pipeline.py         solo encadena y ejecuta
└── medicion.py         solo compara implementaciones
```

En el proyecto modelo, estos cuatro archivos existen en la carpeta `src/` de la raíz y
contienen las mismas clases y funciones que este cuaderno desarrolla paso a paso. La
sección 13 verifica que ambas versiones producen el mismo resultado.

Cada archivo tiene un propósito que se puede enunciar en una frase. Si al describir un
archivo hace falta la palabra «y» varias veces, probablemente tenga cohesión baja.

**Para el informe.** El apartado de diseño estructurado pide justificar la organización.
La prueba del acoplamiento es el mejor argumento: «para cambiar la estrategia de
imputación solo se modifica `transformadores.py`, porque el pipeline no conoce el
detalle de cada paso».

---
# 8. Validación: casos normales, límite y excepciones

La rúbrica pide los tres escenarios. Acá va uno de cada tipo, con el conjunto de Airbnb.

In [25]:
# --- CASO NORMAL: el flujo completo sobre el conjunto real ---
pipe = Pipeline([EliminadorColumnas("id"), ImputadorMediana("beds")])
salida = pipe.ajustar(datos).transformar(datos)

assert len(salida) == len(datos), "El pipeline perdió o duplicó filas"
assert salida["beds"].isna().sum() == 0, "Quedaron nulos en beds"
assert "id" not in salida.columns, "La columna id no se eliminó"
print("Caso normal: las tres comprobaciones pasaron.")

Caso normal: las tres comprobaciones pasaron.


In [26]:
# --- CASOS LÍMITE: situaciones extremas pero válidas ---

# 1) Una columna sin ningún nulo no debe alterarse
sin_nulos = pd.DataFrame({"beds": [4.5, 4.8, 5.0]})
r1 = ImputadorMediana("beds").ajustar_transformar(sin_nulos)
assert r1["beds"].tolist() == [4.5, 4.8, 5.0]
print("Límite 1: columna sin nulos, sin cambios.")

# 2) Una sola categoría genera una sola columna
una_cat = pd.DataFrame({"room_type": ["Entire home/apt", "Entire home/apt", "Entire home/apt"]})
r2 = CodificadorNominal("room_type").ajustar_transformar(una_cat)
print("Límite 2: una categoría ->", list(r2.columns))

# 3) Varianza cero no debe producir división por cero
constante = pd.DataFrame({"minimum_nights": [10.0, 10.0, 10.0]})
r3 = EscaladorRobusto("minimum_nights").ajustar_transformar(constante)
assert r3["minimum_nights"].notna().all()
print("Límite 3: varianza cero ->", r3["minimum_nights"].tolist())

# 4) Una categoría nueva en prueba no debe crear columna
pipe_cat = Pipeline([CodificadorNominal("neighbourhood_cleansed")]).ajustar(entrenamiento)
nuevos = prueba.copy()
# Insertamos una comuna que no existía en el conjunto de entrenamiento
nuevos.iloc[0, nuevos.columns.get_loc("neighbourhood_cleansed")] = "Comuna Ficticia"
r4 = pipe_cat.transformar(nuevos)
print("Límite 4: categoría nueva no crea columna ->",
      "neighbourhood_cleansed_Comuna_Ficticia" not in r4.columns)

Límite 1: columna sin nulos, sin cambios.
Límite 2: una categoría -> ['room_type_Entire_home/apt']
Límite 3: varianza cero -> [0.0, 0.0, 0.0]
Límite 4: categoría nueva no crea columna -> True


In [27]:
# --- EXCEPCIONES: entradas que deben fallar con mensaje claro ---

pruebas = [
    ("transformar sin ajustar", lambda: ImputadorMediana("beds").transformar(datos)),
    ("columna inexistente", lambda: ImputadorMediana("columna_ficticia").ajustar(datos)),
    ("agregar algo que no es Transformador", lambda: Pipeline().agregar("texto")),
    ("pipeline sin ajustar", lambda: Pipeline([ImputadorMediana("beds")]).transformar(datos)),
]

for descripcion, accion in pruebas:
    try:
        accion()
        print(f"  {descripcion:<38} NO lanzó excepción (revisar)")
    except (RuntimeError, KeyError, TypeError) as error:
        print(f"  {descripcion:<38} {type(error).__name__} capturado")

  transformar sin ajustar                RuntimeError capturado
  columna inexistente                    KeyError capturado
  agregar algo que no es Transformador   TypeError capturado
  pipeline sin ajustar                   RuntimeError capturado


## La verificación que más rinde: el pipeline de clases contra la Fase 2

Si el pipeline con clases produce exactamente el mismo conjunto que guardó la Fase 2, hay una prueba objetiva de que la reorganización no rompió nada. Se compara contra `data/processed/airbnb_procesado.csv`, ajustando con todos los datos porque así trabajó la Fase 2. El índice se reinicia porque el One-Hot de la Fase 2 lo reinició.

In [28]:
# Resultado oficial de la Fase 2, tal como quedó guardado en el repositorio
f2_oficial = pd.read_csv(RUTA_F2)

# El mismo trabajo, hecho por el pipeline de clases.
# Se ajusta con TODOS los datos porque así trabajó la F2 (sin separar entrenamiento y prueba)
pipeline_f2 = construir_pipeline().ajustar(datos)
f3_clases = pipeline_f2.transformar(datos).reset_index(drop=True)

print("Fase 2 (archivo guardado):", f2_oficial.shape)
print("Fase 3 (pipeline de clases):", f3_clases.shape)

# Compara filas, columnas, nombres, orden, tipos y valores.
# Si hubiera UNA sola diferencia, se detiene con un error que dice dónde está.
pd.testing.assert_frame_equal(f3_clases, f2_oficial)
print("\nLas dos versiones son idénticas: mismas filas, columnas, tipos y valores.")

Fase 2 (archivo guardado): (14758, 50)
Fase 3 (pipeline de clases): (14758, 50)

Las dos versiones son idénticas: mismas filas, columnas, tipos y valores.


---
# 9. Recursividad

Toda función recursiva tiene dos partes: un **caso base** que la detiene y un **caso
recursivo** que reduce el problema y vuelve a llamarse.

**Cuándo se justifica:** cuando no se sabe de antemano cuán profundo es el problema.
Si la profundidad es fija, un bucle es más simple y más rápido.

In [29]:
def aplanar(estructura, prefijo=""):
    """Convierte metadatos anidados en pares plano de clave y valor.

    Por qué recursión y no un bucle: la profundidad no se conoce al escribir el
    código. Mañana alguien agrega un nivel y el bucle anidado deja de servir.
    """
    plano = {}
    for clave in estructura:
        valor = estructura[clave]
        compuesta = f"{prefijo}.{clave}" if prefijo else str(clave)
        if isinstance(valor, dict) and valor:
            plano.update(aplanar(valor, compuesta))
        else:
            plano[compuesta] = valor
    return plano


# Metadatos adaptados al proyecto de Airbnb
metadatos = {
    "proyecto": {"nombre": "airbnb_santiago", "fase": 3},
    "datos": {"filas": len(datos), "nulos": {"beds": int(datos["beds"].isna().sum())}},
    "entorno": {"semilla": SEMILLA, "librerias": {"pandas": pd.__version__}},
}

for clave, valor in aplanar(metadatos).items():
    print(f"{clave:<32} {valor}")

proyecto.nombre                  airbnb_santiago
proyecto.fase                    3
datos.filas                      18904
datos.nulos.beds                 203
entorno.semilla                  42
entorno.librerias.pandas         3.1.1


## El costo de repetir trabajo

La recursión **no es** sinónimo de eficiencia. Estas dos funciones tienen la misma
idea y un costo muy distinto.

In [30]:
import time
import pandas as pd

def fib_ingenua(n):
    """Sin memoria: recalcula los mismos subproblemas miles de veces."""
    if n < 2:
        return n
    return fib_ingenua(n - 1) + fib_ingenua(n - 2)


def fib_memoizada(n, cache=None):
    """La misma recursión, guardando lo ya calculado."""
    if cache is None:
        cache = {}
    if n in cache:
        return cache[n]
    resultado = n if n < 2 else fib_memoizada(n - 1, cache) + fib_memoizada(n - 2, cache)
    cache[n] = resultado
    return resultado


filas = []
for n in [18, 22, 26]:
    inicio = time.perf_counter(); fib_ingenua(n); t1 = time.perf_counter() - inicio
    inicio = time.perf_counter(); fib_memoizada(n); t2 = time.perf_counter() - inicio
    filas.append({
        "n": n, 
        "ingenua_s": round(t1, 6), 
        "memoizada_s": round(t2, 6),
        "veces_mas_rapida": round(t1 / t2, 1) if t2 > 0 else 0.0
    })

pd.DataFrame(filas)

,n,ingenua_s,memoizada_s,veces_mas_rapida
0,18,0.000357,0.000007,51.7
1,22,0.002863,0.000012,234.6
2,26,0.016535,0.000015,1073.7


El cambio entre una versión y otra es de dos líneas, y el efecto se mide en cientos de
veces. Lo que hace eficiente a la segunda no es la recursión: es **no repetir trabajo**.

**Si su proyecto no tiene un problema recursivo natural**, decláren­lo en el informe y
expliquen por qué un enfoque iterativo es preferible. La rúbrica acepta recursividad
**o** división funcional, y una decisión argumentada vale lo mismo que implementarla.

---
# 10. Eficiencia: medir tiempo y memoria

La rúbrica pide mediciones **reproducibles** con `timeit` o equivalente, comparación
entre implementaciones e interpretación considerando tiempo **y** memoria.

In [31]:
import numpy as np
import pandas as pd
import timeit

def medir(funcion, *args, **kwargs):
    """Ejecuta la función una vez y devuelve (resultado, segundos, memoria pico en MB)."""
    tracemalloc.start()                       # empieza a vigilar la memoria
    inicio = time.perf_counter()
    resultado = funcion(*args, **kwargs)
    segundos = time.perf_counter() - inicio
    _, pico = tracemalloc.get_traced_memory() # el máximo de memoria que se usó
    tracemalloc.stop()
    return resultado, segundos, pico / 1024 / 1024


# Datos reales justo antes de escalar: todos los pasos del pipeline menos los escaladores
pasos_previos = [p for p in construir_pipeline().pasos_ejecutados()
                 if not isinstance(p, EscaladorRobusto)]
antes_de_escalar = datos.copy()
for paso in pasos_previos:
    antes_de_escalar = paso.ajustar_transformar(antes_de_escalar)

escalador = EscaladorRobusto("n_comodidades").ajustar(antes_de_escalar)
MEDIANA, RIC = escalador.parametros["mediana"], escalador.parametros["ric"]


def escalar_con_bucle(serie):
    """Versión A: valor por valor, con un for."""
    escalados = []
    for valor in serie:
        escalados.append((valor - MEDIANA) / RIC)
    return escalados


def escalar_vectorizado(serie):
    """Versión B: toda la columna de una vez (la que usa EscaladorRobusto)."""
    return ((serie - MEDIANA) / RIC).tolist()


serie = antes_de_escalar["n_comodidades"]

# Antes de comparar tiempos: las dos deben dar EXACTAMENTE lo mismo
assert escalar_con_bucle(serie) == escalar_vectorizado(serie), "Las versiones no coinciden"
print(f"Las dos versiones dan el mismo resultado sobre los {len(serie)} alojamientos.\n")

# Se mide 5 veces y se conserva el MENOR tiempo
t_bucle = min(timeit.repeat(lambda: escalar_con_bucle(serie), number=50, repeat=5)) / 50
t_vect = min(timeit.repeat(lambda: escalar_vectorizado(serie), number=50, repeat=5)) / 50
_, _, mem_bucle = medir(escalar_con_bucle, serie)
_, _, mem_vect = medir(escalar_vectorizado, serie)

print(f"A · con bucle   : {t_bucle*1000:.3f} ms   {mem_bucle*1024:.1f} KB")
print(f"B · vectorizada : {t_vect*1000:.3f} ms   {mem_vect*1024:.1f} KB")
print(f"\nLa vectorizada es {t_bucle / t_vect:.1f} veces más rápida "
      f"y usa {mem_vect / mem_bucle:.2f} veces la memoria del bucle.")

Las dos versiones dan el mismo resultado sobre los 14758 alojamientos.

A · con bucle   : 1.586 ms   462.3 KB
B · vectorizada : 0.311 ms   575.2 KB

La vectorizada es 5.1 veces más rápida y usa 1.24 veces la memoria del bucle.


**Cómo se lee.** Las dos versiones hacen la misma cuenta, `(valor − mediana) / RIC`, y el `assert` confirma que el resultado es idéntico. La diferencia está en **quién** hace el trabajo:

- **Con bucle**, Python ejecuta una resta y una división por cada alojamiento, una por una.
- **Vectorizada**, pandas entrega la columna completa a NumPy, que hace todas las cuentas de una vez en código compilado. Por eso es varias veces más rápida.

**El costo está en la memoria.** La versión vectorizada crea primero una columna intermedia completa y recién después la convierte en lista, así que su pico de memoria es mayor. Gana en tiempo y pierde en espacio: ese es el intercambio que la rúbrica pide analizar.

**Cómo crece el costo con el tamaño.** Una medición sobre un solo tamaño no permite hablar de complejidad: hay que ver cómo cambia el tiempo cuando crece el volumen. Hasta el tamaño real se usan muestras del conjunto; sobre él se repiten filas al azar (con reemplazo) solo para observar la tendencia.

In [32]:
# Cómo crece el costo con el tamaño: esto es lo que revela la complejidad
mediciones = []
for n in [1_000, 5_000, len(serie), 50_000, 100_000]:
    # Sobre el tamaño real se repiten filas al azar, solo para ver la tendencia
    muestra = serie.sample(n=n, replace=n > len(serie), random_state=SEMILLA)
    t_b = min(timeit.repeat(lambda: escalar_con_bucle(muestra), number=10, repeat=5)) / 10
    t_v = min(timeit.repeat(lambda: escalar_vectorizado(muestra), number=10, repeat=5)) / 10
    _, _, m_b = medir(escalar_con_bucle, muestra)
    _, _, m_v = medir(escalar_vectorizado, muestra)
    mediciones.append({"filas": n, "bucle_ms": round(t_b*1000, 3),
                       "vectorizada_ms": round(t_v*1000, 3), "razón": round(t_b/t_v, 1),
                       "bucle_KB": round(m_b*1024, 1), "vectorizada_KB": round(m_v*1024, 1)})

print(pd.DataFrame(mediciones).to_string(index=False))

 filas  bucle_ms  vectorizada_ms  razón  bucle_KB  vectorizada_KB
  1000     0.106           0.079    1.3      29.9            37.8
  5000     0.534           0.142    3.8     155.9           194.0
 14758     1.582           0.302    5.2     462.3           575.2
 50000     5.360           0.896    6.0    1603.6          1951.8
100000    10.764           1.682    6.4    3123.8          3905.0


**Cómo se lee la tabla.**

- **Tiempo.** El tiempo del bucle crece en proporción a las filas: con 100 veces más filas, tarda cerca de 100 veces más. Es complejidad temporal **lineal, O(n)**: ningún método evita mirar cada valor una vez. La vectorizada también es O(n); no cambia la forma de la curva, cambia la **constante**: cada valor le cuesta mucho menos.
- **La razón crece con el tamaño.** En tamaños pequeños la ventaja casi desaparece, porque la operación vectorizada tiene un costo fijo de preparación que en pocas filas pesa más que el ahorro. A medida que crecen las filas, ese costo fijo se reparte y la ventaja se estabiliza en varias veces.
- **Memoria.** Las dos crecen en proporción a las filas (complejidad espacial **O(n)**), porque ambas devuelven una lista del mismo largo. La vectorizada usa siempre alrededor de un 25 % más, por la columna intermedia.

**Tres reglas que se cumplieron para que la medición valga**

1. **Se repitió y se conservó el tiempo menor.** `timeit.repeat` mide 5 veces y se toma el mínimo: los valores altos suelen reflejar interrupciones del sistema operativo, no el costo del código.
2. **Se midió sobre el tamaño real**: los 14.758 alojamientos completos que procesa el pipeline, además de tamaños menores y mayores para ver la tendencia.
3. **Se comprobó que las versiones dan lo mismo**, con `assert`, antes de comparar tiempos.

**La decisión que tomó la medición.** `EscaladorRobusto.aplicar()` usa la forma vectorizada. En el tamaño real, la ganancia de tiempo es de varias veces y la memoria extra es del orden de cien kilobytes, una cantidad despreciable frente a la memoria de cualquier computador. El intercambio cambiaría si el conjunto fuera tan grande que no cupiera en memoria: ahí convendría procesarlo por bloques.

---
# 11. Patrones de diseño

Un patrón es una solución conocida a un problema que se repite. Lo importante:
**ustedes ya usaron varios en la Fase 2 sin saber su nombre.**

## Strategy: varias formas de hacer lo mismo

En la Fase 2 comparamos imputar `beds` con la mediana, con la media o por grupo (por ejemplo, según el tipo de habitación o comuna). Si cada forma es un bloque distinto, cambiar de una a otra obliga a reescribir. Strategy separa el **qué** del **cómo**.

In [33]:
class EstrategiaImputacion:
    """Contrato común de todas las estrategias."""
    etiqueta = "sin definir"

    def calcular(self, df, columna):
        raise NotImplementedError


class PorMediana(EstrategiaImputacion):
    etiqueta = "mediana"

    def calcular(self, df, columna):
        return float(df[columna].median())


class PorMedia(EstrategiaImputacion):
    etiqueta = "media"

    def calcular(self, df, columna):
        return float(df[columna].mean())


class PorMedianaDeGrupo(EstrategiaImputacion):
    etiqueta = "mediana por grupo"

    def __init__(self, columna_grupo):
        self.columna_grupo = columna_grupo

    def calcular(self, df, columna):
        return df.groupby(self.columna_grupo)[columna].median().to_dict()


class ImputadorFlexible(Transformador):
    """No sabe imputar: sabe cuándo. El cómo lo aporta la estrategia."""

    def __init__(self, columna, estrategia=None):
        super().__init__(columna)          # llama al constructor de la clase base
        self.estrategia = estrategia or PorMediana()

    def aprender(self, df):
        return {"valor": self.estrategia.calcular(df, self.columna),
                "estrategia": self.estrategia.etiqueta}

    def aplicar(self, df):
        valor = self._parametros["valor"]
        if isinstance(valor, dict):                       # mediana por grupo
            relleno = df[self.estrategia.columna_grupo].map(valor)
            df[self.columna] = df[self.columna].fillna(relleno)
            df[self.columna] = df[self.columna].fillna(df[self.columna].median())
        else:
            df[self.columna] = df[self.columna].fillna(valor)
        return df


# La comparación de estrategias que pide la rúbrica, en un bucle adaptada a Airbnb
desv_original = datos["beds"].std()
comparacion = []

# Agrupamos por 'room_type' (tipo de alojamiento) para la estrategia por grupo
for estrategia in [PorMediana(), PorMedia(), PorMedianaDeGrupo("room_type")]:
    salida = ImputadorFlexible("beds", estrategia).ajustar_transformar(datos)
    comparacion.append({
        "estrategia": estrategia.etiqueta,
        "desv_antes": round(desv_original, 4),
        "desv_despues": round(salida["beds"].std(), 4),
        "cambio_pct": round((salida["beds"].std() - desv_original) / desv_original * 100, 2),
    })

pd.DataFrame(comparacion)

,estrategia,desv_antes,desv_despues,cambio_pct
0,mediana,1.842,1.8321,-0.54
1,media,1.842,1.8320,-0.54
2,mediana por grupo,1.842,1.8328,-0.49


**Nota sobre estos números.** En el archivo completo, `beds` tiene 203 faltantes de 18.904 filas (cerca de un 1 %). Con tan pocos huecos, las tres estrategias cambian la desviación estándar en menos de un 1 % y son prácticamente equivalentes. Es la misma conclusión de la comparación de imputaciones de la Fase 2: cuando las estrategias no se distinguen, se declara y se elige la más simple, que es la mediana, coherente además con la asimetría de las variables del estudio.

Esa tabla es la comparación que el criterio de preprocesamiento pide justificar. Toda
imputación por un valor central reduce la dispersión; lo que se compara es **cuánto**, y
esa cifra es el argumento para elegir.

## Factory: decidir qué construir

La transformación que corresponde depende del rol de la variable. Factory concentra esa
decisión en un solo lugar, en vez de repetirla en cada cuaderno.

In [34]:
def crear_transformador(rol, columna, **parametros):
    """Devuelve el transformador que corresponde al rol analítico."""
    rol = rol.strip().lower()

    if rol == "continua":
        return ImputadorFlexible(columna, parametros.get("estrategia"))
    if rol == "nominal":
        return CodificadorNominal(columna)
    if rol == "escalar":
        return EscaladorRobusto(columna)
    if rol == "identificador":
        return EliminadorColumnas(columna)

    raise ValueError(
        f"Rol desconocido: '{rol}'. "
        "Roles válidos: continua, nominal, escalar, identificador."
    )


# El diccionario de variables de la Fase 1 adaptado al proyecto de Airbnb
diccionario = [
    {"variable": "id", "rol": "identificador"},
    {"variable": "beds", "rol": "continua"},
    {"variable": "room_type", "rol": "nominal"},
    {"variable": "neighbourhood_cleansed", "rol": "nominal"},
    {"variable": "minimum_nights", "rol": "escalar"},
]

pipeline_auto = Pipeline([crear_transformador(v["rol"], v["variable"]) for v in diccionario])
salida = pipeline_auto.ajustar(datos).transformar(datos)

print(pipeline_auto.resumen().to_string(index=False))
print("\nResultado:", salida.shape)

try:
    crear_transformador("geografica", "latitude")
except ValueError as error:
    print("\nValueError:", error)

 orden                                       paso              clase
     1                     EliminadorColumnas(id) EliminadorColumnas
     2                    ImputadorFlexible(beds)  ImputadorFlexible
     3              CodificadorNominal(room_type) CodificadorNominal
     4 CodificadorNominal(neighbourhood_cleansed) CodificadorNominal
     5           EscaladorRobusto(minimum_nights)   EscaladorRobusto

Resultado: (18904, 123)

ValueError: Rol desconocido: 'geografica'. Roles válidos: continua, nominal, escalar, identificador.


## Observer: registrar sin estorbar

La bitácora de decisiones suele escribirse a mano después de ejecutar, con el riesgo de
quedar desactualizada. Con Observer, el pipeline **avisa** cada vez que termina un paso
y quien quiera registrar se suscribe.

In [35]:
class Bitacora:
    """Observador que acumula lo que ocurre y lo entrega como tabla."""

    def __init__(self):
        self.registros = []

    def notificar(self, paso, filas, columnas, segundos, parametros):
        self.registros.append({"paso": paso, "filas": filas, "columnas": columnas,
                               "segundos": round(segundos, 5), "parametros": parametros})

    def a_dataframe(self):
        return pd.DataFrame(self.registros)


class ReporteConsola:
    """Otro observador: imprime mientras ocurre."""

    def notificar(self, paso, filas, columnas, segundos, parametros):
        print(f"  {paso:<34} {filas:>6} filas, {columnas:>3} col  [{segundos:.4f}s]")


class PipelineObservable(Pipeline):
    """Hereda todo de Pipeline y agrega la capacidad de ser observado."""

    def __init__(self, pasos=None):
        super().__init__(pasos)
        self._observadores = []

    def suscribir(self, observador):
        self._observadores.append(observador)
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError("Pipeline: hay que ajustar antes de transformar.")
        resultado = df.copy()
        for paso in self._pasos:
            inicio = time.perf_counter()
            resultado = paso.transformar(resultado)
            transcurrido = time.perf_counter() - inicio
            for observador in self._observadores:
                observador.notificar(paso.nombre, len(resultado), resultado.shape[1],
                                     transcurrido, paso.parametros)
        return resultado


# Instanciamos la bitácora y armamos el pipeline observable adaptado a Airbnb
bitacora = Bitacora()
pipe = PipelineObservable([
    EliminadorColumnas("id"),
    ImputadorFlexible("beds"),
    CodificadorNominal("room_type"),
    EscaladorRobusto("minimum_nights"),
])
pipe.suscribir(bitacora).suscribir(ReporteConsola())

print("Ejecución del pipeline:")
salida = pipe.ajustar(datos).transformar(datos)

print("\nBitácora producida por la propia ejecución:")
bitacora.a_dataframe()[["paso", "filas", "columnas", "segundos"]]

Ejecución del pipeline:
  EliminadorColumnas(id)              18904 filas,  89 col  [0.0075s]
  ImputadorFlexible(beds)             18904 filas,  89 col  [0.0098s]
  CodificadorNominal(room_type)       18904 filas,  92 col  [0.0137s]
  EscaladorRobusto(minimum_nights)    18904 filas,  92 col  [0.0108s]

Bitácora producida por la propia ejecución:


,paso,filas,columnas,segundos
0,EliminadorColumnas(id),18904,89,0.00750
1,ImputadorFlexible(beds),18904,89,0.00980
2,CodificadorNominal(room_type),18904,92,0.01374
3,EscaladorRobusto(minimum_nights),18904,92,0.01079


**Fíjense en lo que acaba de pasar.** La bitácora no se escribió a mano: se genera cada
vez que el pipeline corre, así que no puede quedar desactualizada.

## Singleton: una sola configuración

Rutas, semilla y umbrales se necesitan en varios módulos. Si cada uno construye su
propia configuración, dos partes del pipeline pueden usar semillas distintas sin que
nadie lo note.

**La advertencia:** Singleton introduce estado global, que complica las pruebas. Úsenlo
solo para configuración y pasen el resto de los parámetros explícitamente.

In [36]:
class Configuracion:
    """Una sola instancia compartida por todo el proyecto."""

    _instancia = None

    def __new__(cls, *args, **kwargs):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            cls._instancia._iniciada = False
        return cls._instancia

    def __init__(self, semilla=SEMILLA):
        if self._iniciada:               # evita reiniciar una instancia existente
            return
        self.semilla = semilla
        self.parametros = {}
        self._iniciada = True

    def definir(self, clave, valor):
        self.parametros[clave] = valor
        return self


# Instanciación y prueba del Singleton adaptado a Airbnb
a = Configuracion(semilla=SEMILLA)
b = Configuracion(semilla=99)            # se ignora: la instancia ya existe

# Definimos parámetros propios de tu pipeline de alojamientos
a.definir("umbral_nulos", 0.6).definir("test_size", 0.2).definir("objetivos", OBJETIVOS)

print("¿Son el mismo objeto?", a is b)
print("Semilla vista desde b:", b.semilla)
print("Parámetros desde b   :", b.parametros)

¿Son el mismo objeto? True
Semilla vista desde b: 42
Parámetros desde b   : {'umbral_nulos': 0.6, 'test_size': 0.2, 'objetivos': ['price', 'estimated_occupancy_l365d']}


## Resumen: qué patrón usar y cuándo

| Patrón | El problema que resuelve | Dónde ya lo hicieron en la Fase 2 |
|---|---|---|
| **Strategy** | Varias formas de hacer lo mismo | Comparar imputaciones de `bmi` |
| **Factory** | Elegir qué construir según una condición | Decidir la transformación según el rol |
| **Observer** | Registrar sin que el ejecutor sepa quién anota | La bitácora de decisiones |
| **Singleton** | Una sola configuración compartida | La semilla y las rutas del proyecto |

**No usen los cuatro.** Usen el que su proyecto necesita y expliquen por qué. Un patrón
aplicado donde no hace falta agrega complejidad sin beneficio, y eso también se evalúa.

---
---
# 12. El pipeline construido desde la configuración

El pipeline de la sección 6.1 no tiene ningún nombre de columna escrito a mano: todo sale de la celda de configuración. Es Factory aplicado al proyecto completo: la configuración deja de ser documentación y pasa a gobernar el código.

In [37]:
pipeline_config = construir_pipeline()
print(f"Pipeline construido con {len(pipeline_config)} pasos desde la configuración:\n")
print(pipeline_config.resumen().to_string(index=False))

Pipeline construido con 24 pasos desde la configuración:

 orden                                       paso                clase
     1                     ConversorPrecio(price)      ConversorPrecio
     2   FiltroFilas(solo alojamientos completos)          FiltroFilas
     3             SelectorColumnas(11 variables)     SelectorColumnas
     4                   Acotador(maximum_nights)             Acotador
     5       FiltroFilas(objetivos sin faltantes)          FiltroFilas
     6                    MarcadorNulos(bedrooms)        MarcadorNulos
     7             ImputadorMediana(accommodates)     ImputadorMediana
     8                 ImputadorMediana(latitude)     ImputadorMediana
     9                 ImputadorMediana(bedrooms)     ImputadorMediana
    10                     ImputadorMediana(beds)     ImputadorMediana
    11           ImputadorMediana(minimum_nights)     ImputadorMediana
    12           ImputadorMediana(maximum_nights)     ImputadorMediana
    13          Fil

In [38]:
# Se ajusta con entrenamiento y se aplica a prueba, sin fuga de datos
pipeline_config.ajustar(entrenamiento)
entrenamiento_listo = pipeline_config.transformar(entrenamiento)
prueba_lista = pipeline_config.transformar(prueba)

print("Entrenamiento:", entrenamiento.shape, "->", entrenamiento_listo.shape)
print("Prueba       :", prueba.shape, "->", prueba_lista.shape)

# Las dos salidas deben tener exactamente las mismas columnas y en el mismo orden
assert list(entrenamiento_listo.columns) == list(prueba_lista.columns), \
    "Entrenamiento y prueba quedaron con columnas distintas"
print("\nAmbos conjuntos tienen las mismas columnas, en el mismo orden.")

Entrenamiento: (15123, 90) -> (11788, 50)
Prueba       : (3781, 90) -> (2970, 50)

Ambos conjuntos tienen las mismas columnas, en el mismo orden.


**Esa última comprobación es más importante de lo que parece.** Si el codificador
aprendiera el vocabulario de cada conjunto por separado, entrenamiento y prueba
quedarían con columnas distintas y cualquier modelo posterior fallaría. Como el
vocabulario se aprende una sola vez en el ajuste, eso no puede pasar.

## Guardar el resultado y su configuración

El conjunto procesado no sirve de mucho sin el registro de cómo se produjo. Guardar
ambos juntos es lo que hace el trabajo trazable.

In [39]:
def guardar_resultado(df, ruta_salida, pipeline, carpeta=DIR_PROCESADO):
    """Escribe el conjunto procesado y un registro de los parámetros aprendidos."""
    os.makedirs(carpeta, exist_ok=True)
    destino = os.path.join(carpeta, ruta_salida)
    df.to_csv(destino, index=False)

    registro = pd.DataFrame([
        {"orden": i, "paso": p.nombre, "parametros": str(p.parametros)}
        for i, p in enumerate(pipeline.pasos_ejecutados(), start=1)
    ])
    registro.to_csv(destino.replace(".csv", "_parametros.csv"), index=False)

    print(f"Conjunto guardado   : {destino} ({df.shape[0]} filas x {df.shape[1]} col)")
    print(f"Parámetros guardados: {destino.replace('.csv', '_parametros.csv')}")
    return registro


registro = guardar_resultado(entrenamiento_listo, "entrenamiento_procesado.csv",
                             pipeline_config)
registro.head()

Conjunto guardado   : c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\entrenamiento_procesado.csv (11788 filas x 50 col)
Parámetros guardados: c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\entrenamiento_procesado_parametros.csv


,orden,paso,parametros
0,1,ConversorPrecio(price),"{'regla': ""quitar '$' y ',' y convertir a núme..."
1,2,FiltroFilas(solo alojamientos completos),{'regla': 'solo alojamientos completos'}
2,3,SelectorColumnas(11 variables),{'conservadas': 11}
3,4,Acotador(maximum_nights),"{'maximo': 1125, 'valores_sobre_el_maximo': 2}"
4,5,FiltroFilas(objetivos sin faltantes),{'regla': 'objetivos sin faltantes'}


---
# 13. El resultado: cómo queda el código y cómo quedan los datos

Esta sección cierra el recorrido. Muestra dos cosas: **cómo se ve el proyecto después de
aplicar POO** y **qué conjunto queda listo para la Fase 4**.

## 13.1 El código, antes y después

**Antes (Fase 2).** El pipeline era una secuencia de celdas con funciones. Funcionaba, pero el orden estaba implícito y cada paso calculaba sus parámetros sobre el conjunto completo:

```python
df_crudo["price"] = pd.to_numeric(df_crudo["price"].str.replace("$", "")...)
df = df_crudo[df_crudo["room_type"] == "Entire home/apt"][variables_estudio]
df["maximum_nights"] = df["maximum_nights"].clip(upper=1125)
df = df.dropna(subset=OBJETIVOS)
for col in columnas_imputar:
    df = imputar_nulos_numericos(df, col, estrategia="mediana")
...
df = codificar_one_hot(df, "neighbourhood_cleansed", nombres_comunas)
df, scaler = escalar_caracteristicas(df, columnas_a_escalar)
```

**Después (Fase 3).** El mismo trabajo, en cuatro líneas:

```python
pipeline = construir_pipeline()
pipeline.ajustar(entrenamiento)
entrenamiento_listo = pipeline.transformar(entrenamiento)
prueba_lista = pipeline.transformar(prueba)
```

Lo que cambió no es el resultado (la sección 8 lo comprueba): ahora cada paso es una pieza con nombre, que se puede probar, reemplazar y reutilizar sin tocar las demás, y que aprende sus parámetros solo del conjunto de entrenamiento.

## 13.2 La arquitectura, generada desde el propio código

En vez de escribir a mano la tabla de arquitectura para el informe, se puede producir
leyendo las clases que existen. Así no queda desactualizada.

In [40]:
def documentar_arquitectura():
    """Genera la tabla de arquitectura a partir de las clases definidas.

    __subclasses__() devuelve las clases que heredan de Transformador. Es la
    misma información que Python usa para resolver la herencia, así que la
    tabla refleja el código real y no lo que creemos que hay.
    """
    filas = [{
        "componente": "Transformador",
        "rol": "clase base",
        "responsabilidad": "Define el contrato y controla el estado de cada paso",
        "archivo sugerido": "src/transformadores.py",
    }]

    for clase in Transformador.__subclasses__():
        resumen = (clase.__doc__ or "sin documentar").strip().split("\n")[0]
        filas.append({
            "componente": clase.__name__,
            "rol": "clase hija",
            "responsabilidad": resumen,
            "archivo sugerido": "src/transformadores.py",
        })

    filas += [
        {"componente": "Pipeline", "rol": "orquestador",
         "responsabilidad": "Encadena los pasos y controla el orden de ajuste",
         "archivo sugerido": "src/pipeline.py"},
        {"componente": "PipelineObservable", "rol": "orquestador",
         "responsabilidad": "Pipeline que además notifica cada paso a sus observadores",
         "archivo sugerido": "src/pipeline.py"},
        {"componente": "Bitacora", "rol": "observador",
         "responsabilidad": "Registra lo ocurrido en cada paso de la ejecución",
         "archivo sugerido": "src/observadores.py"},
        {"componente": "construir_pipeline", "rol": "fábrica",
         "responsabilidad": "Arma el pipeline desde la celda de configuración",
         "archivo sugerido": "src/fabrica.py"},
        {"componente": "medir", "rol": "utilidad",
         "responsabilidad": "Mide tiempo y memoria de cualquier función",
         "archivo sugerido": "src/medicion.py"},
    ]
    return pd.DataFrame(filas)


arquitectura = documentar_arquitectura()
print(arquitectura.to_string(index=False))

          componente         rol                                                            responsabilidad       archivo sugerido
       Transformador  clase base                       Define el contrato y controla el estado de cada paso src/transformadores.py
    ImputadorMediana  clase hija                                Rellena los nulos con la mediana aprendida. src/transformadores.py
  CodificadorNominal  clase hija                       Convierte una columna de categorías en columnas 0/1. src/transformadores.py
    EscaladorRobusto  clase hija Centra en la mediana y divide por el rango intercuartílico (RobustScaler). src/transformadores.py
  EliminadorColumnas  clase hija          Quita columnas que no aportan al análisis, como el identificador. src/transformadores.py
     ConversorPrecio  clase hija                      Convierte el precio de texto ("$55,106.00") a número. src/transformadores.py
         FiltroFilas  clase hija                         Conserva solo las filas qu

**Esa tabla es el apartado de documentación de arquitectura del informe.** Cópienla,
agreguen dos o tres párrafos explicando **por qué** esa división, y el criterio queda
cubierto.

La estructura de archivos que sugiere la última columna es la que corresponde llevar al
repositorio. Los módulos van en la carpeta `src/` de la raíz, compartida por todas las
fases; en la carpeta de la fase queda solo el cuaderno:

```
proyecto-grupoX-mcdi500/
├── data/processed/                  lo que se entrega a la Fase 4
├── src/
│   ├── carga.py                     lectura y verificación del archivo
│   ├── transformadores.py           la clase base y sus hijas
│   ├── pipeline.py                  Pipeline
│   └── medicion.py                  medir()
└── F3/
    └── S2_F3_NucleoAlgoritmico_Eficiencia_POO.ipynb   este cuaderno, ejecutado
```

`PipelineObservable`, `Bitacora` y la fábrica son extensiones opcionales: si su proyecto
las usa, van en `src/` con el mismo criterio (por ejemplo `src/observadores.py`).


## 13.3 El conjunto procesado: qué queda listo

Acá se ejecuta el pipeline completo y se revisa el resultado con el detalle que la
Fase 4 va a necesitar.

In [41]:
# Pipeline definitivo, construido desde la configuración
pipeline_final = construir_pipeline()
pipeline_final.ajustar(entrenamiento)

entrenamiento_final = pipeline_final.transformar(entrenamiento)
prueba_final = pipeline_final.transformar(prueba)

print("CONJUNTO PROCESADO")
print("=" * 58)
print(f"Entrenamiento : {entrenamiento_final.shape[0]:>6} filas x {entrenamiento_final.shape[1]:>3} columnas")
print(f"Prueba        : {prueba_final.shape[0]:>6} filas x {prueba_final.shape[1]:>3} columnas")
print(f"Partida desde : {datos.shape[0]:>6} filas x {datos.shape[1]:>3} columnas")
print(f"\nColumnas finales: {entrenamiento_final.shape[1]} (las mismas 50 de la Fase 2)")
entrenamiento_final.head(3)

CONJUNTO PROCESADO
Entrenamiento :  11788 filas x  50 columnas
Prueba        :   2970 filas x  50 columnas
Partida desde :  18904 filas x  90 columnas

Columnas finales: 50 (las mismas 50 de la Fase 2)


,price,estimated_occupancy_l365d,accommodates,bedrooms,beds,latitude,minimum_nights,maximum_nights,bedrooms_imputado,n_comodidades,...,comuna_Pudahuel,comuna_Quilicura,comuna_Quinta_Normal,comuna_Recoleta,comuna_Renca,comuna_San_Joaquín,comuna_San_Miguel,comuna_Santiago,comuna_Vitacura,comuna_Ñuñoa
2388,84947.0,36,0.5,1.0,0.5,-0.512463,0.0,1.0,0,0.454545,...,0,0,0,0,0,0,0,0,0,1
9680,90000.0,0,-0.5,0.0,-0.5,1.139384,3.0,0.0,0,-1.227273,...,0,0,0,0,0,0,0,0,1,0
4489,43500.0,0,-0.5,0.0,-0.5,-0.451679,-1.0,1.0,0,0.727273,...,0,0,0,0,0,0,0,1,0,0


In [42]:
# Esquema final: qué tipo tiene cada columna y de dónde salió
def describir_esquema(df, originales):
    """Clasifica cada columna del resultado según su origen."""
    filas = []
    for columna in df.columns:
        if columna in originales:
            origen = "original"
        elif "_" in columna and columna.split("_")[0] in originales:
            origen = f"derivada de {columna.split('_')[0]}"
        else:
            origen = "derivada"
        filas.append({
            "columna": columna,
            "tipo": str(df[columna].dtype),
            "origen": origen,
            "nulos": int(df[columna].isna().sum()),
        })
    return pd.DataFrame(filas)


esquema = describir_esquema(entrenamiento_final, set(datos.columns))
print(f"Total de columnas: {len(esquema)}")
print(f"Originales conservadas: {(esquema['origen'] == 'original').sum()}")
print(f"Derivadas de la codificación: {(esquema['origen'] != 'original').sum()}")
print(f"Columnas con nulos: {(esquema['nulos'] > 0).sum()}")
print()
esquema.head(12)

Total de columnas: 50
Originales conservadas: 8
Derivadas de la codificación: 42
Columnas con nulos: 0



,columna,tipo,origen,nulos
0,price,float64,original,0
1,estimated_occupancy_l365d,int64,original,0
2,accommodates,float64,original,0
3,bedrooms,float64,original,0
4,beds,float64,original,0
5,latitude,float64,original,0
6,minimum_nights,float64,original,0
7,maximum_nights,float64,original,0
8,bedrooms_imputado,int64,derivada de bedrooms,0
9,n_comodidades,float64,derivada,0


## 13.3 bis Del cuaderno a los módulos (pendiente)

El criterio de diseño estructurado no se cumple con clases escritas dentro del cuaderno: el código reutilizable debe vivir en `src/` y el cuaderno debe importarlo. **Este paso queda pendiente**: hoy `src/` solo contiene `utilidades.py`. La celda siguiente revisa qué módulos faltan, sin detener el cuaderno.

In [43]:
# Módulos que la arquitectura propone para src/ (sección 13.2)
modulos_esperados = ["carga.py", "transformadores.py", "pipeline.py", "medicion.py"]
carpeta_src = RAIZ / "src"

for modulo in modulos_esperados:
    estado = "existe" if (carpeta_src / modulo).exists() else "PENDIENTE"
    print(f"  src/{modulo:<20} {estado}")

  src/carga.py             PENDIENTE
  src/transformadores.py   PENDIENTE
  src/pipeline.py          PENDIENTE
  src/medicion.py          PENDIENTE


## 13.4 Verificación de entrega

Antes de pasar a la Fase 4 conviene comprobar cinco condiciones. Si alguna falla, el
problema es de esta fase y corregirlo acá cuesta mucho menos que descubrirlo después.

In [44]:
def verificar_entrega(entrenamiento_final, prueba_final, total_final, objetivos):
    """Cinco comprobaciones antes de entregar el conjunto a la Fase 4."""
    controles = []

    # 1. La partición no perdió ni duplicó filas: entrenamiento + prueba = total procesado
    obtenidas = len(entrenamiento_final) + len(prueba_final)
    controles.append(("Partición sin filas perdidas ni duplicadas",
                      obtenidas == len(total_final), f"{obtenidas} de {len(total_final)}"))

    # 2. Entrenamiento y prueba tienen el mismo esquema
    mismo_esquema = list(entrenamiento_final.columns) == list(prueba_final.columns)
    controles.append(("Mismo esquema en ambos conjuntos",
                      mismo_esquema, f"{entrenamiento_final.shape[1]} columnas"))

    # 3. No quedan nulos en ninguno de los dos conjuntos
    nulos = int(entrenamiento_final.isna().sum().sum() + prueba_final.isna().sum().sum())
    controles.append(("Sin valores nulos", nulos == 0, f"{nulos} nulos"))

    # 4. Los objetivos sobrevivieron completos
    objetivos_ok = all(o in entrenamiento_final.columns
                       and entrenamiento_final[o].isna().sum() == 0 for o in objetivos)
    controles.append(("Objetivos presentes y completos", objetivos_ok, ", ".join(objetivos)))

    # 5. Todo quedó numérico y sin identificador
    numerico = entrenamiento_final.select_dtypes(exclude="number").empty
    sin_id = COLUMNA_ID not in entrenamiento_final.columns
    controles.append(("Todo numérico y sin identificador", numerico and sin_id, COLUMNA_ID))

    tabla = pd.DataFrame(
        [{"control": c[0], "estado": "OK" if c[1] else "FALLA", "detalle": c[2]}
         for c in controles]
    )
    return tabla, all(c[1] for c in controles)


total_final = pipeline_final.transformar(datos)
controles, aprobado = verificar_entrega(entrenamiento_final, prueba_final,
                                        total_final, OBJETIVOS)
print(controles.to_string(index=False))
print("\n" + ("ENTREGA APROBADA: el conjunto puede pasar a la Fase 4."
               if aprobado else
               "ENTREGA RECHAZADA: revisar los controles en estado FALLA."))

                                   control estado                          detalle
Partición sin filas perdidas ni duplicadas     OK                   14758 de 14758
          Mismo esquema en ambos conjuntos     OK                      50 columnas
                         Sin valores nulos     OK                          0 nulos
           Objetivos presentes y completos     OK price, estimated_occupancy_l365d
         Todo numérico y sin identificador     OK                               id

ENTREGA APROBADA: el conjunto puede pasar a la Fase 4.


## 13.5 Lo que se entrega a la Fase 4

La Fase 4 no debe volver a limpiar ni transformar. Recibe cuatro artefactos y trabaja
sobre ellos.

In [45]:
def entregar_a_fase4(entrenamiento_final, prueba_final, pipeline, esquema,
                     carpeta=DIR_PROCESADO):
    """Escribe los cuatro artefactos que recibe la Fase 4."""
    os.makedirs(carpeta, exist_ok=True)
    entregables = {}

    # 1. Los conjuntos listos para analizar
    ruta_ent = os.path.join(carpeta, "entrenamiento_f3.csv")
    ruta_pru = os.path.join(carpeta, "prueba_f3.csv")
    entrenamiento_final.to_csv(ruta_ent, index=False)
    prueba_final.to_csv(ruta_pru, index=False)
    entregables["entrenamiento"] = ruta_ent
    entregables["prueba"] = ruta_pru

    # 2. El diccionario del conjunto resultante
    ruta_dic = os.path.join(carpeta, "diccionario_f3.csv")
    esquema.to_csv(ruta_dic, index=False)
    entregables["diccionario"] = ruta_dic

    # 3. Los parámetros aprendidos: es lo que permite reproducir el resultado
    ruta_par = os.path.join(carpeta, "parametros_f3.csv")
    pd.DataFrame([
        {"orden": i, "paso": p.nombre, "parametros": str(p.parametros)}
        for i, p in enumerate(pipeline.pasos_ejecutados(), start=1)
    ]).to_csv(ruta_par, index=False)
    entregables["parametros"] = ruta_par

    return entregables


entregables = entregar_a_fase4(entrenamiento_final, prueba_final,
                               pipeline_final, esquema)

print("ARTEFACTOS ENTREGADOS A LA FASE 4")
print("=" * 58)
for nombre, ruta in entregables.items():
    tamano = os.path.getsize(ruta) / 1024
    print(f"  {nombre:<16} {ruta:<40} {tamano:>7.1f} KB")

ARTEFACTOS ENTREGADOS A LA FASE 4
  entrenamiento    c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\entrenamiento_f3.csv  1806.6 KB
  prueba           c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\prueba_f3.csv   456.1 KB
  diccionario      c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\diccionario_f3.csv     1.7 KB
  parametros       c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\parametros_f3.csv     2.3 KB


### El contrato entre fases

| Artefacto | Qué contiene | Para qué lo usa la Fase 4 |
|---|---|---|
| `entrenamiento_f3.csv` | El conjunto procesado de entrenamiento | Ajustar y explorar |
| `prueba_f3.csv` | El conjunto procesado de prueba | Evaluar, con el mismo esquema |
| `diccionario_f3.csv` | Columna, tipo, origen y nulos | Saber qué significa cada variable |
| `parametros_f3.csv` | Lo que aprendió cada paso | Reproducir el resultado o auditarlo |

**Lo que la Fase 4 NO debe hacer:** volver a imputar, volver a codificar ni volver a
escalar. Si lo hiciera sobre el conjunto ya transformado, escalaría dos veces y los
resultados dejarían de tener sentido.

**Lo que sí debe hacer:** leer estos archivos, explorar, visualizar y comunicar. Y si
necesita una transformación nueva, agregarla como un paso más del pipeline de la Fase 3,
no como código suelto en su propio cuaderno. Esa es la ventaja de haber modularizado.

In [46]:
# Comprobación final: lo que se guardó se puede volver a leer sin sorpresas
releido = pd.read_csv(entregables["entrenamiento"])

print("Verificación de ida y vuelta")
print(f"  Antes de guardar : {entrenamiento_final.shape}")
print(f"  Después de leer  : {releido.shape}")
print(f"  Mismas columnas  : {list(releido.columns) == list(entrenamiento_final.columns)}")
print(f"  Sin nulos nuevos : {int(releido.isna().sum().sum()) == int(entrenamiento_final.isna().sum().sum())}")
print("\nEl conjunto está listo para la Fase 4.")

Verificación de ida y vuelta
  Antes de guardar : (11788, 50)
  Después de leer  : (11788, 50)
  Mismas columnas  : True
  Sin nulos nuevos : True

El conjunto está listo para la Fase 4.


---
# 14. Su turno: ejercicios para el proyecto

Estos cinco ejercicios producen directamente la evidencia que pide la rúbrica.
Resuélvanlos con **su propio conjunto de datos**, no con el de stroke.

## Ejercicio 1 · Encapsular un paso de su pipeline
**Alimenta:** criterio de programación orientada a objetos (6 pts).

In [47]:
class MiTransformador(Transformador):
    """TODO: describir qué hace, sobre qué columna y por qué."""

    def aprender(self, df):
        # TODO: calcular lo que hay que aprender del conjunto de entrenamiento
        # Devuelve un diccionario con los parámetros
        return {}

    def aplicar(self, df):
        # TODO: aplicar la transformación usando self._parametros
        return df


# TODO: probarla con su conjunto
# mi_paso = MiTransformador("su_columna")
# resultado = mi_paso.ajustar_transformar(su_df)
# print(mi_paso.nombre, mi_paso.parametros)

## Ejercicio 2 · Armar su pipeline completo
**Alimenta:** diseño estructurado (6 pts) y preprocesamiento (6 pts).

In [48]:
# TODO: reemplazar por los pasos de SU pipeline de la Fase 2
mi_pipeline = Pipeline([
    # EliminadorColumnas("su_identificador"),
    # ImputadorFlexible("su_columna_con_nulos"),
    # CodificadorNominal("su_categorica"),
    # EscaladorRobusto("su_continua"),
])

# TODO: ajustar con entrenamiento y transformar prueba
# entrenamiento = su_df.sample(frac=0.8, random_state=SEMILLA)
# prueba = su_df.drop(index=entrenamiento.index)
# mi_pipeline.ajustar(entrenamiento)
# resultado = mi_pipeline.transformar(prueba)

## Ejercicio 3 · Comparar dos implementaciones
**Alimenta:** eficiencia y optimización (6 pts).

In [49]:
def implementacion_a(df):
    # TODO: una forma, por ejemplo con bucle sobre filas
    pass


def implementacion_b(df):
    # TODO: otra forma, por ejemplo vectorizada
    pass


# TODO: comprobar primero que dan el mismo resultado
# assert implementacion_a(su_df) == implementacion_b(su_df)

# TODO: medir con timeit y con medir() para la memoria
# TODO: repetir sobre varios tamaños para ver cómo crece la diferencia
# TODO: interpretar el resultado en una celda de texto

## Ejercicio 4 · Validar en tres escenarios
**Alimenta:** validación técnica (6 pts).

In [50]:
# TODO: CASO NORMAL
# assert len(resultado) == len(su_df), "El pipeline perdió o duplicó filas"

# TODO: CASO LÍMITE
# Ideas: columna sin nulos, una sola categoría, varianza cero,
#        categoría que aparece solo en prueba

# TODO: EXCEPCIÓN
# try:
#     MiTransformador("columna_inexistente").ajustar(su_df)
# except KeyError as error:
#     print("KeyError capturado:", error)

## Ejercicio 5 · Elegir un patrón y verificar que nada se rompió
**Alimenta:** documentación de arquitectura (6 pts) y validación técnica (6 pts).

In [51]:
# TODO 1: elegir UN patrón de la tabla resumen e implementarlo.
#         Escribir tres frases: qué problema resolvía, cómo lo resuelve
#         el patrón, y qué habría pasado sin él.

# TODO 2: comparar el conjunto procesado de la Fase 2 contra el que
#         produce su pipeline de clases
# anterior = pd.read_csv("../F2/data/processed/su_archivo.csv")
# nuevo = mi_pipeline.ajustar(su_df).transformar(su_df)
# pd.testing.assert_frame_equal(anterior, nuevo)
# print("La reorganización no alteró el resultado")

---
# Cierre

## Lo que se llevan

| Concepto | Dónde se ve en este cuaderno |
|---|---|
| Encapsulamiento | `_parametros`, `_ajustado`, propiedades de solo lectura |
| Herencia | `ImputadorMediana`, `CodificadorNominal` y `EscaladorRobusto` heredan de `Transformador` |
| Polimorfismo | El `Pipeline` recorre los pasos sin preguntar su tipo |
| Recursividad | `aplanar()` y el contraste entre Fibonacci ingenua y memoizada |
| Eficiencia | `timeit`, `perf_counter`, `tracemalloc` y la tabla de crecimiento |
| Patrones | Strategy, Factory, Observer y Singleton |

## Antes de la próxima sesión

1. Conviertan **un** paso de su pipeline en una clase que herede de `Transformador`.
2. Midan **una** operación de dos formas distintas, con la comprobación de equivalencia.
3. Decidan si su proyecto justifica recursividad, y anoten por qué.
4. Ejecuten `assert_frame_equal` contra su conjunto de la Fase 2.

Con esos cuatro pasos tienen el esqueleto de la entrega.

## Bibliografía (APA 7)

Gamma, E., Helm, R., Johnson, R., & Vlissides, J. (1994). *Design patterns: Elements of
reusable object-oriented software*. Addison-Wesley.

McKinney, W. (2022). *Python for data analysis* (3.ª ed.). O'Reilly Media.

Python Software Foundation. (s. f.). *Classes*. https://docs.python.org/3/tutorial/classes.html

Python Software Foundation. (s. f.). *timeit — Measure execution time of small code
snippets*. https://docs.python.org/3/library/timeit.html

The pandas development team. (s. f.). *pandas documentation*. https://pandas.pydata.org/docs/

---

*Avance Fase 3 · Semana 2 · MCDI500 · Magíster en Ciencia de Datos e Inteligencia
Artificial · Universidad Andrés Bello*